# Análise de dados aplicada ao contexto de apostas no futebol

## Integrantes:
* Bruno Basckeira Chinaglia
* Douglas da Fontoura Pereyra

## Objetivo da Apresentação
Esta apresentação detalha o processo de coleta, limpeza, análise e clusterização de dados de odds de futebol de diversas ligas mundiais. Nosso foco é identificar padrões, semelhanças e anomalias nas características das partidas, tanto a nível de ligas/temporadas quanto de partidas individuais.


## ⚽ Seção 1: Coleta de Dados - Web Scraping

Nesta seção, o código Python utiliza a biblioteca Selenium para realizar a raspagem de dados (web scraping) do site 'Oddspedia'. O objetivo é coletar informações de partidas de futebol de uma determinada liga (no caso, a Bundesliga) ao longo das últimas 10 temporadas, incluindo placares e odds de apostas.

### **Perguntas Relevantes à Coleta:**

*   **Quando foi coletado?** A coleta visa dados de temporadas específicas, que neste caso foram os anos de 2015 a 2025.

*   **Que viés carrega?** O principal viés aqui é inerente à fonte dos dados (Oddspedia) e à natureza das odds. Odds de apostas refletem a percepção do mercado sobre a probabilidade de um evento, e não necessariamente a probabilidade 'real'. Podem ser influenciadas por volumes de apostas, informações públicas, lesões de jogadores, etc. Além disso, a coleta manual ou semi-automática pode introduzir vieses se o site mudar sua estrutura ou se houver falhas na raspagem de dados.

*   **Quem já usou/onde tá disponibilizado?** Este script foi desenvolvido para uso interno neste projeto. Os dados brutos, uma vez coletados, são armazenados localmente e, eventualmente, disponibilizados para as etapas subsequentes de análise.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

# ---- Configuração do Selenium ----
options = webdriver.ChromeOptions()
# options.add_argument("--headless")  # Descomente para depuração

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# ---- Lista para armazenar dados ----
dados = []

# URL base para Bundesliga
url_base = "https://oddspedia.com/br/futebol/alemanha/bundesliga"
driver.get(url_base)
time.sleep(5)  # Espera a página carregar

def close_overlays(driver):
    # Fecha popups/overlays comuns
    try:
        # Exemplo: fecha popups de cookies
        cookie_btns = driver.find_elements(By.CSS_SELECTOR, '[id*="cookie"] button, .cookie-popup__btn, .cookie-popup__close')
        for btn in cookie_btns:
            if btn.is_displayed():
                try:
                    btn.click()
                    time.sleep(0.5)
                except Exception:
                    pass
        # Fecha overlays genéricos
        overlays = driver.find_elements(By.CSS_SELECTOR, '.overlay, .modal, .popup, .close, .btn-close')
        for overlay in overlays:
            if overlay.is_displayed():
                try:
                    overlay.click()
                    time.sleep(0.5)
                except Exception:
                    pass
    except Exception:
        pass

# Lista de temporadas a buscar (ordem crescente, split seasons)
anos = ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

for ano in anos:
    print(f"\n==== Temporada {ano} ====")
    formatos = [str(ano)]
    ano_selecionado = False
    for formato in formatos:
        try:
            close_overlays(driver)
            dropdown_div = driver.find_element(By.CSS_SELECTOR, ".old-dropdown.content__header--league__dropdown.old-dropdown--small.old-dropdown--dark")
            dropdown_btn = dropdown_div.find_element(By.CSS_SELECTOR, ".old-dropdown__toggle")
            driver.execute_script("arguments[0].scrollIntoView();", dropdown_btn)
            driver.execute_script("arguments[0].click();", dropdown_btn)
            time.sleep(1)
            year_options = dropdown_div.find_elements(By.XPATH, f".//div[contains(@class,'old-dropdown__list-item') and normalize-space(text())='{formato}']")
            if year_options:
                driver.execute_script("arguments[0].click();", year_options[0])
                time.sleep(3)
                ano_selecionado = True
                print(f"Selecionado: {formato}")
                break
        except Exception as e:
            print(f"Falha ao tentar selecionar '{formato}': {e}")
    if not ano_selecionado:
        print(f"Ano {ano} não encontrado no dropdown de temporada.")
        continue

    semana = 1
    while True:
        print(f"Coletando {ano} - Semana {semana}")

        # Coletar todas as partidas da semana
        partidas = driver.find_elements(By.CSS_SELECTOR, ".match-list-item")
        for partida in partidas:
            try:
                # Times
                times = partida.find_elements(By.CSS_SELECTOR, ".match-team__name")
                time_casa = times[0].text.strip()
                time_fora = times[1].text.strip()

                # Placar (gols usando estrutura detalhada)
                score_div = partida.find_element(By.CSS_SELECTOR, ".old-match-score")
                gols_casa = int(score_div.find_element(By.CSS_SELECTOR, ".old-match-score__team--home .old-match-score-result__score").text.strip())
                gols_fora = int(score_div.find_element(By.CSS_SELECTOR, ".old-match-score__team--away .old-match-score-result__score").text.strip())

                # Odds
                odds = partida.find_elements(By.CSS_SELECTOR, ".odd-box-with-logo__value")
                odd_casa = float(odds[0].text.strip())
                odd_empate = float(odds[1].text.strip())
                odd_fora = float(odds[2].text.strip())

                # Vencedor (a odd com class "won")
                vencedor = None
                odd_vencedora = None
                odds_divs = partida.find_elements(By.CSS_SELECTOR, ".odd-box-with-logo")
                for i, div in enumerate(odds_divs):
                    if "odd-box-with-logo--won" in div.get_attribute("class"):
                        if i == 0:
                            vencedor, odd_vencedora = time_casa, odd_casa
                        elif i == 1:
                            vencedor, odd_vencedora = "Empate", odd_empate
                        else:
                            vencedor, odd_vencedora = time_fora, odd_fora

                dados.append({
                    "temporada": ano,
                    "temporada_dropdown": formato if ano_selecionado else None,
                    "semana": semana,
                    "time_casa": time_casa,
                    "time_fora": time_fora,
                    "gols_casa": gols_casa,
                    "gols_fora": gols_fora,
                    "odd_casa": odd_casa,
                    "odd_empate": odd_empate,
                    "odd_fora": odd_fora,
                    "vencedor": vencedor,
                    "odd_vencedora": odd_vencedora
                })
            except Exception as e:
                print("Erro ao processar partida:", e)

        # Tentar clicar no botão "próxima semana"
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, ".ml-pagination__btn--next")
            if "disabled" in next_btn.get_attribute("class"):
                print(f"Fim da temporada {ano}")
                break
            next_btn.click()
            semana += 1
            time.sleep(3)
        except Exception:
            print(f"Fim da temporada {ano} (sem botão next)")
            break

# Fechar navegador
driver.quit()

# ---- Salvar em CSV ----
df = pd.DataFrame(dados)
df.to_csv("odds_Bundesliga_2015_2025.csv", index=False, encoding="utf-8")
print("Coleta finalizada e salva em odds_Bundesliga_2015_2025.csv")

## 🧹 Seção 2: Pré-processamento e Limpeza de Dados

Após a coleta inicial, é crucial limpar e preparar os dados para análise. Esta seção do código realiza diversas operações para garantir a qualidade e a consistência do conjunto de dados, que para este exemplo foi carregado de um arquivo CSV de 'odds_brasileirao_2015_2024.csv'.

### **Principais etapas:**
1.  **Cálculo de `goal_diff` e `winner_computed`**: Deriva a diferença de gols e o vencedor de cada partida a partir dos placares.
2.  **Verificação de consistência**: Compara o `winner_computed` com a coluna `vencedor` já existente (se houver) para identificar inconsistências nos dados.
3.  **Remoção de duplicatas**: Garante que cada partida única seja representada apenas uma vez no conjunto de dados.
4.  **Ordenação**: Organiza os dados por temporada e semana para facilitar análises sequenciais.
5.  **Criação de `odd_favorito` e `odd_preterido`**: Identifica a menor e a maior odd para cada partida.
6.  **Remoção de colunas redundantes/auxiliares**: Descarta colunas temporárias ou não necessárias para a análise final.

### **Perguntas Relevantes ao Pré-processamento:**

*   **Qual número de exemplos no conjunto de dados?** Após o pré-processamento (e a deduplicação), o conjunto de dados `odds_brasileirao_2015_2024_cleaned.csv` contém **3699** registros (partidas), conforme o log de saída. Cada registro é um 'exemplo' ou 'instância' de uma partida de futebol.

*   **Conjunto desbalanceado? O que foi feito pra amenizar?**
    Não lidamos com desbalanceamento de classes, mas fizemos um pré-processamento para lidar com dados faltantes, esses dados puderam ser preenchidos através dos valores de outros atributos. Estamos buscando *padrões intrínsecos* nos dados.

    O log de `missing values` mostra que a coluna `vencedor` tinha 261 valores ausentes, que foram preenchidos com base no `goal_diff`. Este preenchimento garante a completude dos dados para a análise, mas não 'ameniza' um desbalanceamento de classes real; apenas resolve a falta de rótulos.

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

file_path = Path("odds_brasileirao_2015_2024.csv")
df = pd.read_csv(file_path)
# Calcula goal_diff e winner_computed ANTES da deduplicação
if 'gols_casa' in df.columns and 'gols_fora' in df.columns:
    df['goal_diff'] = df['gols_casa'] - df['gols_fora']
    def computed_winner(row):
        if pd.isna(row['gols_casa']) or pd.isna(row['gols_fora']):
            return np.nan
        if row['gols_casa'] > row['gols_fora']:
            return row.get('time_casa')
        elif row['gols_casa'] < row['gols_fora']:
            return row.get('time_fora')
        else:
            return 'Empate'
    df['winner_computed'] = df.apply(computed_winner, axis=1)
    # Verifica a consistência com a coluna 'vencedor' existente, se houver
    if 'vencedor' in df.columns:
        df['winner_consistent'] = df.apply(lambda r: (pd.isna(r['vencedor']) and pd.isna(r['winner_computed'])) or (str(r.get('vencedor')).strip() == str(r.get('winner_computed')).strip()), axis=1)

# Remove duplicatas exatas (mesma temporada, semana, times e placar)
before = len(df)
dedup_cols = [c for c in ['temporada','semana','time_casa','time_fora','gols_casa','gols_fora'] if c in df.columns]
if dedup_cols:
    df = df.drop_duplicates(subset=dedup_cols)
after = len(df)
print(f"Foram removidas {before-after} linhas duplicadas")

# Ordena as linhas por temporada e semana, se disponíveis
sort_cols = []
if 'temporada' in df.columns:
    sort_cols.append('temporada')
if 'semana' in df.columns:
    sort_cols.append('semana')
if sort_cols:
    df = df.sort_values(by=sort_cols).reset_index(drop=True)

# Adiciona odd_favorito (menor odd) e odd_preterido (maior odd)
odds_cols = [c for c in ['odd_casa', 'odd_fora'] if c in df.columns]
if odds_cols:
    df['odd_favorito'] = df[odds_cols].min(axis=1)
    df['odd_preterido'] = df[odds_cols].max(axis=1)
    print("Colunas adicionadas: odd_favorito, odd_preterido")

# Remove colunas auxiliares/redundantes (NÃO mantenha winner_consistent no arquivo final limpo)
cols_to_drop = [c for c in ['temporada_norm', 'season_start', 'winner_computed', 'temporada_dropdown', 'winner_consistent'] if c in df.columns]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Colunas redundantes removidas: {cols_to_drop}")

# Salva o CSV limpo
out_path = file_path.parent / f"{file_path.stem}_cleaned.csv"
df.to_csv(out_path, index=False, encoding='utf-8')
print(f"Dados limpos salvos em: {out_path} (shape: {df.shape})")
print(f"\nColunas finais: {df.columns.tolist()}")
display(df.head(5))

In [ ]:

# ---- Exibe o CSV limpo ----
import pandas as pd
from pathlib import Path

cleaned_file = Path(out_path)

if cleaned_file.exists():
    df_cleaned = pd.read_csv(cleaned_file, encoding='utf-8')
    print(f"\n=== Dados Limpos do Brasileirão ===")
    print(f"Shape: {df_cleaned.shape}")
    print(f"\nNomes das colunas:\n{df_cleaned.columns.tolist()}\n")
    print(f"Tipos de dados:\n{df_cleaned.dtypes}\n")
    print(f"Primeiras 10 linhas:\n")
    display(df_cleaned.head(10))
    print(f"\nÚltimas 5 linhas:\n")
    display(df_cleaned.tail(5))
    print(f"\nEstatísticas básicas:\n")
    display(df_cleaned.describe())
    print(f"\nValores ausentes:\n{df_cleaned.isnull().sum()}\n")
else:
    print(f"Arquivo limpo não encontrado em {cleaned_file}. Por favor, execute a célula de pré-processamento primeiro.")

In [ ]:

# ---- Analisa e corrige vencedor/odd_vencedora ausentes (apenas QA) ----
import pandas as pd
from pathlib import Path
import numpy as np

cleaned_file = Path(out_path)

if cleaned_file.exists():
    df = pd.read_csv(cleaned_file, encoding='utf-8')

    print(f"Valores ausentes ANTES da correção:")
    print(df.isnull().sum())

    # Corrige vencedor e odd_vencedora ausentes com base no goal_diff
    for idx, row in df.iterrows():
        if pd.isna(row.get('vencedor')) or pd.isna(row.get('odd_vencedora')):
            goal_diff = row.get('goal_diff', 0)
            if goal_diff > 0:
                df.at[idx, 'vencedor'] = row.get('time_casa')
                df.at[idx, 'odd_vencedora'] = row.get('odd_casa')
            elif goal_diff == 0:
                df.at[idx, 'vencedor'] = 'Empate'
                df.at[idx, 'odd_vencedora'] = row.get('odd_empate')
            else:  # goal_diff < 0
                df.at[idx, 'vencedor'] = row.get('time_fora')
                df.at[idx, 'odd_vencedora'] = row.get('odd_fora')

    print(f"\nValores ausentes APÓS a correção:")
    print(df.isnull().sum())

    # Para QA: calcula winner_consistent em memória, mas NÃO o persiste no CSV limpo
    def computed_winner(row):
        if pd.isna(row.get('gols_casa')) or pd.isna(row.get('gols_fora')):
            return np.nan
        if row['gols_casa'] > row['gols_fora']:
            return row.get('time_casa')
        elif row['gols_casa'] < row['gols_fora']:
            return row.get('time_fora')
        else:
            return 'Empate'

    df['winner_computed'] = df.apply(computed_winner, axis=1)
    df['winner_consistent'] = df.apply(
        lambda r: (pd.isna(r.get('vencedor')) and pd.isna(r.get('winner_computed'))) or
                  (str(r.get('vencedor')).strip() == str(r.get('winner_computed')).strip()),
        axis=1
    )

    # Mostra a análise de consistência (apenas QA)
    true_count = int(df['winner_consistent'].sum()) if 'winner_consistent' in df.columns else 0
    false_count = int((~df['winner_consistent']).sum()) if 'winner_consistent' in df.columns else 0
    null_count = int(df['winner_consistent'].isnull().sum()) if 'winner_consistent' in df.columns else 0
    total = len(df)

    print(f"\n=== Análise de Consistência do Vencedor (em memória, não salvo) ===")
    print(f"Total de linhas: {total}")
    print(f"winner_consistent = True:  {true_count} ({100*true_count/total:.2f}%)")
    print(f"winner_consistent = False: {false_count} ({100*false_count/total:.2f}%)")
    print(f"winner_consistent = NaN:   {null_count} ({100*null_count/total:.2f}%)")

    if false_count > 0:
        print(f"\nLinhas inconsistentes restantes (primeiras 10):")
        inconsistent = df[df['winner_consistent'] == False][['temporada', 'semana', 'time_casa', 'time_fora', 'gols_casa', 'gols_fora', 'vencedor']].head(10)
        display(inconsistent)

    # Remove colunas QA temporárias antes de salvar para que o CSV limpo NÃO contenha 'winner_consistent'
    for tmp_col in ['winner_computed', 'winner_consistent']:
        if tmp_col in df.columns:
            df = df.drop(columns=[tmp_col])

    # Salva o CSV corrigido (sem winner_consistent)
    df.to_csv(cleaned_file, index=False, encoding='utf-8')
    print(f"\nDados corrigidos salvos em: {cleaned_file} (winner_consistent removido antes de salvar)")

else:
    print(f"Arquivo limpo não encontrado em {cleaned_file}. Por favor, execute a célula de pré-processamento primeiro.")

# Reabre o arquivo limpo e fornece um breve resumo de QA (recalcula em memória se necessário)
if cleaned_file.exists():
    df_cleaned = pd.read_csv(cleaned_file, encoding='utf-8')
    print(f"\nApós salvar, colunas do CSV limpo: {df_cleaned.columns.tolist()}")
    # Recalcula winner_consistent para um QA rápido na tela (não salva)
    if 'gols_casa' in df_cleaned.columns and 'gols_fora' in df_cleaned.columns:
        df_cleaned['winner_computed'] = df_cleaned.apply(computed_winner, axis=1)
        df_cleaned['winner_consistent'] = df_cleaned.apply(
            lambda r: (pd.isna(r.get('vencedor')) and pd.isna(r.get('winner_computed'))) or
                      (str(r.get('vencedor')).strip() == str(r.get('winner_computed')).strip()),
            axis=1
        )
        true_count = int(df_cleaned['winner_consistent'].sum())
        false_count = int((~df_cleaned['winner_consistent']).sum())
        null_count = int(df_cleaned['winner_consistent'].isnull().sum())
        total = len(df_cleaned)
        print(f"\nQA Rápido (temporário) - winner_consistent: True={true_count}, False={false_count}, NaN={null_count} (total={total})")
        if false_count > 0:
            display(df_cleaned[df_cleaned['winner_consistent'] == False][['temporada', 'semana', 'time_casa', 'time_fora', 'gols_casa', 'gols_fora', 'vencedor']].head(10))
        # remove colunas QA temporárias
        df_cleaned = df_cleaned.drop(columns=['winner_computed', 'winner_consistent'])
    else:
        print("Não há colunas suficientes para recalcular winner_consistent para QA.")

In [ ]:
import pandas as pd
from pathlib import Path

cleaned_file = Path(out_path)

if cleaned_file.exists():
    df_cleaned = pd.read_csv(cleaned_file, encoding='utf-8')
    if 'odd_vencedora' in df_cleaned.columns and 'temporada' in df_cleaned.columns:
        # Converte temporada para ano de início para filtragem
        # Lida com ambos os formatos: "2023" e "16/17" (torna-se 2016)
        def get_start_year(temp_str):
            temp_str = str(temp_str).strip()
            if '/' in temp_str:
                # Formato como "16/17" -> extrai a primeira parte e converte para ano completo
                year_part = temp_str.split('/')[0]
                year_int = int(year_part)
                # Assume que anos de 2 dígitos < 50 são anos 2000, >= 50 são anos 1900
                return 2000 + year_int if year_int < 50 else 1900 + year_int
            else:
                # Formato como "2023"
                return int(temp_str)

        df_cleaned['temp_start_year'] = df_cleaned['temporada'].apply(get_start_year)

        # Filtra para 2023 e posteriores
        df_filtered = df_cleaned[df_cleaned['temp_start_year'] >= 2023]
        if len(df_filtered) > 0:
            max_odd = df_filtered['odd_vencedora'].max()
            print(f"Maior odd_vencedora (2023 em diante): {max_odd}")
            print("Linha(s) com a maior odd_vencedora de 2023 em diante:")
            display(df_filtered[df_filtered['odd_vencedora'] == max_odd][['temporada', 'semana', 'time_casa', 'time_fora', 'gols_casa', 'gols_fora', 'vencedor', 'odd_vencedora']])
        else:
            print("Nenhum dado encontrado para 2023 em diante.")
    else:
        print("Coluna 'odd_vencedora' ou 'temporada' não encontrada no CSV limpo.")
else:
    print(f"Arquivo limpo não encontrado em {cleaned_file}. Por favor, execute a célula de pré-processamento primeiro.")

## 📈 Seção 3: Análise Exploratória - Visualização da `odd_vencedora`

Nesta etapa, utilizamos box plots para visualizar a distribuição da `odd_vencedora` ao longo das temporadas e de forma geral. Isso nos ajuda a identificar a dispersão, a mediana e a presença de outliers nas odds que realmente venceram.

### **Principais Insights:**
*   **Box Plot por Temporada:** Mostra como a distribuição das odds vencedoras pode variar de um ano para outro, com algumas temporadas apresentando maior dispersão ou valores extremos.
*   **Box Plot Geral:** Oferece uma visão consolidada, destacando a faixa mais comum de `odd_vencedora` e a presença de valores atípicos (outliers). As estatísticas detalhadas (média, desvio padrão, quartis, min/max) complementam a análise visual.

In [ ]:
pip install seaborn

In [ ]:
# ---- Box plots de odd_vencedora ----
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

cleaned_file = Path(out_path)

if cleaned_file.exists():
    df_cleaned = pd.read_csv(cleaned_file, encoding='utf-8')

    if 'odd_vencedora' in df_cleaned.columns and 'temporada' in df_cleaned.columns:
        # Cria figura com subplots
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Box plot por temporada
        sns.boxplot(data=df_cleaned, x='temporada', y='odd_vencedora', ax=axes[0])
        axes[0].set_title('Box Plot de odd_vencedora por Temporada', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Temporada', fontsize=12)
        axes[0].set_ylabel('odd_vencedora', fontsize=12)
        axes[0].tick_params(axis='x', rotation=45)
        axes[0].grid(True, alpha=0.3)

        # Box plot geral
        sns.boxplot(data=df_cleaned, y='odd_vencedora', ax=axes[1], color='skyblue')
        axes[1].set_title('Box Plot de odd_vencedora (Todas as Temporadas)', fontsize=14, fontweight='bold')
        axes[1].set_ylabel('odd_vencedora', fontsize=12)
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Imprime estatísticas
        print("\n=== Estatísticas de odd_vencedora ===")
        print(f"\nEstatísticas Gerais:")
        print(df_cleaned['odd_vencedora'].describe())

        print(f"\n\nEstatísticas por Temporada:")
        print(df_cleaned.groupby('temporada')['odd_vencedora'].describe())
    else:
        print("Coluna 'odd_vencedora' ou 'temporada' não encontrada no CSV limpo.")
else:
    print(f"Arquivo limpo não encontrado em {cleaned_file}. Por favor, execute a célula de pré-processamento primeiro.")

## 📊 Seção 4: Análise da Razão `odd_vencedora / odd_favorito`

Complementando a análise anterior, esta seção explora a razão entre a `odd_vencedora` e a `odd_favorito` (a menor odd oferecida na partida). Esta razão (`odd_ratio`) indica o quão surpreendente foi o resultado em comparação com a expectativa do mercado. Um `odd_ratio` alto significa que a equipe/resultado vencedor era considerado improvável (uma "zebra").

### **Principais Insights:**
*   Os box plots da `odd_ratio` por temporada e no geral fornecem uma medida padronizada do grau de "zebra" nos resultados.
*   Um valor de 1 na `odd_ratio` indica que o favorito (menor odd) realmente venceu ou que a odd vencedora era a menor disponível. Valores acima de 1 indicam que a "zebra" aconteceu.
*   As estatísticas (média, desvio padrão, etc.) da `odd_ratio` confirmam a presença de resultados surpreendentes, com valores máximos significativamente acima de 1 (por exemplo, até 16.24 na temporada 2022 do Brasileirão).

In [ ]:
# ---- Box plots de odd_vencedora / odd_favorito (razão) ----
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

cleaned_file = Path(out_path)

if cleaned_file.exists():
    df_cleaned = pd.read_csv(cleaned_file, encoding='utf-8')

    if 'odd_vencedora' in df_cleaned.columns:
        # Garante que odd_favorito exista; calcula se estiver faltando
        if 'odd_favorito' not in df_cleaned.columns:
            odds_cols = [c for c in ['odd_casa', 'odd_empate', 'odd_fora'] if c in df_cleaned.columns]
            if odds_cols:
                df_cleaned['odd_favorito'] = df_cleaned[odds_cols].min(axis=1)
            else:
                df_cleaned['odd_favorito'] = np.nan

        # Calcula a razão e limpa infinitos
        df_cleaned['odd_ratio'] = df_cleaned['odd_vencedora'] / df_cleaned['odd_favorito']
        df_cleaned['odd_ratio'] = df_cleaned['odd_ratio'].replace([np.inf, -np.inf], np.nan)

        # Cria figura com subplots para a razão
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Box plot por temporada (razão)
        sns.boxplot(data=df_cleaned, x='temporada', y='odd_ratio', ax=axes[0])
        axes[0].set_title('Box Plot de odd_vencedora / odd_favorito por Temporada', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Temporada', fontsize=12)
        axes[0].set_ylabel('odd_vencedora / odd_favorito', fontsize=12)
        axes[0].tick_params(axis='x', rotation=45)
        axes[0].grid(True, alpha=0.3)

        # Box plot geral (razão)
        sns.boxplot(data=df_cleaned, y='odd_ratio', ax=axes[1], color='skyblue')
        axes[1].set_title('Box Plot de odd_vencedora / odd_favorito (Todas as Temporadas)', fontsize=14, fontweight='bold')
        axes[1].set_ylabel('odd_vencedora / odd_favorito', fontsize=12)
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Imprime estatísticas para a razão
        print("\n=== Estatísticas de odd_vencedora / odd_favorito (odd_ratio) ===")
        print(f"\nEstatísticas Gerais:")
        print(df_cleaned['odd_ratio'].describe())

        if 'temporada' in df_cleaned.columns:
            print(f"\n\nEstatísticas por Temporada (odd_ratio):")
            print(df_cleaned.groupby('temporada')['odd_ratio'].describe())
    else:
        print("Coluna 'odd_vencedora' não encontrada no CSV limpo.")
else:
    print(f"Arquivo limpo não encontrado em {cleaned_file}. Por favor, execute a célula de pré-processamento primeiro.")

## 🔍 Seção 5: Identificação de Outliers e Correlação de Features

Esta seção aprofunda a análise dos dados, focando na identificação de partidas *outliers* e na exploração das correlações entre as features numéricas. Outliers são partidas com características incomuns, que podem representar "zebras" ou eventos inesperados.

### **Principais etapas:**
1.  **Normalização de `goal_diff`**: Converte a diferença de gols para valores absolutos, focando na magnitude da diferença, não na direção.
2.  **Cálculo da `odd_ratio`**: A razão `odd_vencedora / odd_favorito` é usada como a principal métrica para identificar outliers.
3.  **Detecção de Outliers (Método IQR)**: Partidas cujos valores de `odd_ratio` caem fora do intervalo interquartil (IQR) são classificadas como outliers. São `332` outliers (8.98%) no dataset do Brasileirão.
4.  **Armazenamento de Outliers**: Os dados das partidas outliers são separados e salvos em um arquivo CSV (`odds_brasileirao_2015_2024_cleaned_outliers.csv`).
5.  **Análise de Correlação**: Matrizes de correlação absoluta (heatmap) são geradas para os dados 'normais' e 'outliers' separadamente. Isso revela como as variáveis se relacionam em cada grupo.

### **Tratamento dos Outliers:**
*   **O que foi feito pra amenizar (outliers)?** Os outliers não foram 'amenizados' ou removidos no sentido de limpeza de dados aqui. Pelo contrário, eles foram *identificados e isolados* para uma análise mais aprofundada. O objetivo é entender suas características e se formam grupos distintos, em vez de tratá-los como 'ruído' a ser descartado. A análise de correlação separada para outliers e dados normais já é uma forma de 'amenizar' seu impacto na análise geral, ao mesmo tempo em que os estuda como fenômenos específicos.

In [ ]:
pip install scikit-learn

In [ ]:
# ---- Identifica outliers e cria matrizes de correlação ----
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

cleaned_file = Path(out_path)

if cleaned_file.exists():
    df_cleaned = pd.read_csv(cleaned_file, encoding='utf-8')

    # Converte goal_diff negativo para positivo (valores absolutos)
    if 'goal_diff' in df_cleaned.columns:
        df_cleaned['goal_diff'] = df_cleaned['goal_diff'].abs()
        print("Convertido goal_diff para valores absolutos (valores negativos multiplicados por -1)")

    if 'odd_vencedora' in df_cleaned.columns:
        # Se odd_favorito não estiver presente (segurança), calcula a partir das colunas de odds disponíveis
        if 'odd_favorito' not in df_cleaned.columns:
            odds_cols = [c for c in ['odd_casa', 'odd_empate', 'odd_fora'] if c in df_cleaned.columns]
            if odds_cols:
                df_cleaned['odd_favorito'] = df_cleaned[odds_cols].min(axis=1)
            else:
                df_cleaned['odd_favorito'] = np.nan

        # Calcula a razão: odd_vencedora / odd_favorito - isso ajuda a normalizar entre as partidas
        df_cleaned['odd_ratio'] = df_cleaned['odd_vencedora'] / df_cleaned['odd_favorito']
        # Limpa infinitos e problemas de divisão extrema
        df_cleaned['odd_ratio'] = df_cleaned['odd_ratio'].replace([np.inf, -np.inf], np.nan)

        # Calcula outliers usando o método IQR na razão
        Q1 = df_cleaned['odd_ratio'].quantile(0.25)
        Q3 = df_cleaned['odd_ratio'].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        print(f"\n=== Detecção de Outliers (Método IQR) em odd_vencedora/odd_favorito ===")
        print(f"Q1 (25º percentil): {Q1:.4f}")
        print(f"Q3 (75º percentil): {Q3:.4f}")
        print(f"IQR: {IQR:.4f}")
        print(f"Limite inferior: {lower_bound:.4f}")
        print(f"Limite superior: {upper_bound:.4f}")

        # Separa outliers e dados normais com base na razão
        is_outlier = (df_cleaned['odd_ratio'] < lower_bound) | (df_cleaned['odd_ratio'] > upper_bound)
        df_outliers = df_cleaned[is_outlier].copy()
        df_normal = df_cleaned[~is_outlier].copy()

        print(f"\nTotal de linhas: {len(df_cleaned)}")
        print(f"Dados normais: {len(df_normal)} ({100*len(df_normal)/len(df_cleaned):.2f}%)")
        print(f"Outliers: {len(df_outliers)} ({100*len(df_outliers)/len(df_cleaned):.2f}%)")

        # Exibe a tabela de outliers (inclui a razão e a odd favorita)
        print(f"\n=== Tabela de Outliers ===")
        display(df_outliers[['temporada', 'semana', 'time_casa', 'time_fora',
                             'gols_casa', 'gols_fora', 'odd_casa', 'odd_empate',
                             'odd_fora', 'odd_favorito', 'odd_vencedora', 'odd_ratio', 'goal_diff']].sort_values('odd_ratio', ascending=False))

        # Salva outliers em CSV (inclui odd_ratio)
        outliers_path = cleaned_file.parent / f"{cleaned_file.stem}_outliers.csv"
        df_outliers.to_csv(outliers_path, index=False, encoding='utf-8')
        print(f"\nOutliers salvos em: {outliers_path}")

        # Seleciona colunas numéricas para correlação (exclui 'semana' e 'temporada')
        numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns.tolist()
        # Remove colunas numéricas não-feature e razão da análise de correlação
        for col in ['semana', 'temporada', 'odd_ratio']:
            if col in numeric_cols:
                numeric_cols.remove(col)
        print(f"\nColunas numéricas para correlação: {numeric_cols}")

        # Calcula correlações nos dados ORIGINAIS (não escalados)
        corr_all = df_cleaned[numeric_cols].corr().abs() if numeric_cols else None
        corr_normal = df_normal[numeric_cols].corr().abs() if numeric_cols and len(df_normal) > 0 else None
        corr_outliers = df_outliers[numeric_cols].corr().abs() if numeric_cols and len(df_outliers) > 1 else None

        # Cria matrizes de correlação com escala 0-1 (correlações absolutas)
        # Deriva um nome de liga legível do nome do arquivo limpo
        league_name = cleaned_file.stem.replace('_cleaned', '').replace('odds_', '').replace('_', ' ').title()

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        plt.suptitle(f"{league_name} — Matrizes de Correlação", fontsize=14, fontweight='bold')

        # Matriz de correlação para dados normais
        if corr_normal is not None:
            sns.heatmap(corr_normal, annot=True, fmt='.2f', cmap='coolwarm', vmin=0, vmax=1, ax=axes[0],
                        cbar_kws={'label': 'Correlação Absoluta (0 a 1)'})
        else:
            axes[0].text(0.5, 0.5, 'Nenhuma coluna numérica disponível\npara correlação', ha='center', va='center', fontsize=12)
        axes[0].set_title('Dados Normais', fontsize=12, fontweight='bold')

        # Matriz de correlação para outliers
        if corr_outliers is not None:
            sns.heatmap(corr_outliers, annot=True, fmt='.2f', cmap='coolwarm', vmin=0, vmax=1, ax=axes[1],
                        cbar_kws={'label': 'Correlação Absoluta (0 a 1)'})
            axes[1].set_title('Outliers', fontsize=12, fontweight='bold')
        else:
            axes[1].text(0.5, 0.5, 'Não há outliers suficientes\npara matriz de correlação', ha='center', va='center', fontsize=12)
            axes[1].set_title('Outliers', fontsize=12, fontweight='bold')

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

        # Imprime estatísticas detalhadas dos dados originais
        print(f"\n=== Estatísticas dos Dados (Escala Original) ===")
        print(f"\nEstatísticas de Todos os Dados:")
        if numeric_cols:
            print(df_cleaned[numeric_cols].describe())
        else:
            print("Nenhuma coluna numérica para descrever.")

        print(f"\nEstatísticas dos Dados Normais:")
        if numeric_cols:
            print(df_normal[numeric_cols].describe())
        else:
            print("Nenhuma coluna numérica para descrever.")

        if len(df_outliers) > 0:
            print(f"\nEstatísticas dos Outliers:")
            if numeric_cols:
                print(df_outliers[numeric_cols].describe())
            else:
                print("Não há colunas numéricas ou linhas de outliers suficientes para descrever.")
    else:
        print("Coluna 'odd_vencedora' não encontrada no CSV limpo.")
else:
    print(f"Arquivo limpo não encontrado em {cleaned_file}. Por favor, execute a célula de pré-processamento primeiro.")

## 🌐 Seção 6: Introdução à Clusterização

Com os dados limpos, pré-processados e os outliers identificados, passamos para a etapa de **Clusterização**. Este é um método de aprendizado não supervisionado que busca agrupar objetos similares sem ter rótulos pré-definidos.

### **Paradigma de Aprendizagem:**
Estamos usando o paradigma de **Aprendizado Não Supervisionado**. A razão é que não temos uma variável alvo (classe) para prever; em vez disso, queremos descobrir estruturas e padrões intrínsecos nos dados, agrupando ligas, temporadas ou partidas com características semelhantes. Utilizamos o Algoritmo K-Means, analisando a Silhouette Score para a quantidade ideal de clusters.

### **Nossas Perguntas de Análise (Baseadas nas 3 Perguntas da Apresentação):**

As principais perguntas que buscamos responder com esta análise de clusterização são:
1.  **Como as diferentes ligas e suas temporadas se agrupam com base nas características médias de suas partidas?** (Existem "estilos" de liga ou temporada?)
2.  **Dentro das partidas consideradas "outliers" (as zebras), existem subgrupos distintos de eventos surpreendentes?** (Quais são os tipos de "zebras"?)
3.  **Qual a proximidade (similaridade) entre as ligas, tanto em seus padrões gerais quanto nos seus eventos atípicos (outliers)?** (Quais ligas são mais parecidas ou diferentes?)

In [ ]:
# ---- Análise para Recomendação de Algoritmo de Clusterização ----
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Encontra arquivo limpo automaticamente
data_dir = Path(".")
cleaned_files = list(data_dir.glob("*_cleaned.csv"))

if cleaned_files:
    cleaned_file = cleaned_files[0]  # Usa o primeiro (ou único) arquivo limpo encontrado
    print(f"Usando arquivo: {cleaned_file.name}\n")
    df_cleaned = pd.read_csv(cleaned_file, encoding='utf-8')

    print("="*70)
    print("ANÁLISE DE DADOS PARA RECOMENDAÇÃO DE CLUSTERIZAÇÃO")
    print("="*70)

    # 1. Análise da estrutura
    print("\n1. ESTRUTURA DOS DADOS:")
    print(f"   - Total de registros: {len(df_cleaned)}")
    print(f"   - Total de colunas: {len(df_cleaned.columns)}")
    print(f"   - Colunas numéricas: {df_cleaned.select_dtypes(include=[np.number]).shape[1]}")

    # 2. Identifica colunas numéricas para clusterização
    numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns.tolist()
    for col in ['semana', 'temporada', 'odd_ratio']:
        if col in numeric_cols:
            numeric_cols.remove(col)

    print(f"\n2. COLUNAS DISPONÍVEIS PARA CLUSTERING:")
    print(f"   {numeric_cols}")

    # 3. Verifica valores ausentes
    print(f"\n3. VALORES AUSENTES NOS FEATURES:")
    missing = df_cleaned[numeric_cols].isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("   Sem valores ausentes")

    # 4. Características estatísticas
    print(f"\n4. CARACTERÍSTICAS ESTATÍSTICAS:")
    stats = df_cleaned[numeric_cols].describe()
    print(f"\n   Médias e desvios-padrão (por coluna):")
    for col in numeric_cols:
        print(f"   - {col}: média={stats[col]['mean']:.4f}, std={stats[col]['std']:.4f}")

    # 5. Dimensionalidade
    print(f"\n5. DIMENSIONALIDADE:")
    print(f"   - Número de features: {len(numeric_cols)}")
    print(f"   - Razão (amostras/features): {len(df_cleaned) / len(numeric_cols):.2f}x")

    # 6. Distribuição dos dados
    print(f"\n6. DISTRIBUIÇÃO DOS DADOS:")
    for col in numeric_cols:
        skewness = df_cleaned[col].skew()
        kurtosis = df_cleaned[col].kurtosis()
        print(f"   - {col}: skewness={skewness:.3f}, kurtosis={kurtosis:.3f}")

    # 7. Estrutura temporal
    if 'temporada' in df_cleaned.columns:
        print(f"\n7. ESTRUTURA TEMPORAL (Ligas/Temporadas):")
        leagues = df_cleaned['temporada'].unique()
        print(f"   - Número de ligas/temporadas: {len(leagues)}")
        print(f"   - Temporadas: {sorted(leagues)}")
        for league in sorted(leagues):
            count = len(df_cleaned[df_cleaned['temporada'] == league])
            print(f"     • {league}: {count} registros")

## 🥅 Seção 7: Clusterização K-Means de Ligas Individuais (Temporadas)

Nesta etapa, aplicamos o algoritmo K-Means para agrupar as *temporadas* de uma **única liga** (neste caso, `argentina` foi usada como exemplo no log) com base em suas características médias (gols, odds, etc.). Isso nos ajuda a identificar se há padrões ou "eras" distintas dentro da história de uma liga.

### **Escolha do Algoritmo (K-Means):**
*   **Por que escolhemos K-Means?** Optamos pelo K-Means por sua simplicidade, eficiência computacional e facilidade de interpretação para dados numéricos contínuos. É um bom ponto de partida para a clusterização quando a forma dos clusters é razoavelmente esperada como esférica.

### **Determinação do `k` Ótimo:**
*   Utilizamos dois métodos heurísticos:
    *   **Método do Cotovelo (Elbow Method):** Busca um ponto de inflexão na curva de inércia (soma das distâncias quadradas dos pontos aos seus centroides).
    *   **Silhouette Score:** Mede o quão similar um objeto é ao seu próprio cluster em comparação com outros clusters. Valores mais próximos de 1 indicam boa clusterização.
*   **Resultado do `k` Ótimo:** O Silhouette Score sugeriu `k=3` para a liga argentina de exemplo, indicando 3 grupos distintos de temporadas.

### **Resultado Obtido:**
*   As 10 temporadas da liga argentina foram divididas em 3 clusters:
    *   **Cluster 0:** Temporadas 2022, 2023, 2024
    *   **Cluster 1:** Temporadas 16/17, 17/18, 18/19, 19/20, 2015
    *   **Cluster 2:** Temporadas 2016, 2021
*   A visualização PCA em 2D mostra a separação dos clusters e os centroides. A variância explicada pelos dois primeiros componentes PCA foi de 65.34%, indicando que capturam uma boa parte da informação.

### **É bom?**
*   O Silhouette Score de `0.1959` é um valor baixo, o que sugere que os clusters não são muito bem separados ou densos. Isso pode indicar que as temporadas da liga argentina não possuem distinções tão fortes nos features utilizados, ou que K-Means pode não ser o algoritmo ideal para capturar as nuances dessa liga. Seria interessante explorar as características médias de cada cluster para entender o que os diferencia, apesar do baixo score.

In [ ]:
# ---- Clusterização K-Means - Ligas ----
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# Encontra arquivo limpo automaticamente
data_dir = Path(".")
cleaned_files = list(data_dir.glob("*_cleaned.csv"))

if cleaned_files:
    cleaned_file = cleaned_files[0]
    print(f"Usando arquivo: {cleaned_file.name}\n")
    df = pd.read_csv(cleaned_file, encoding='utf-8')

    # Seleciona colunas numéricas para clusterização (exclui semana, temporada, odd_ratio)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in ['semana', 'temporada', 'odd_ratio']:
        if col in numeric_cols:
            numeric_cols.remove(col)

    print("="*70)
    print("K-MEANS CLUSTERIZAÇÃO - ANÁLISE DE LIGAS")
    print("="*70)
    print(f"\nFeatures para clusterização: {numeric_cols}")
    print(f"Total de registros: {len(df)}")

    # Prepara os dados: agrega por liga (temporada)
    print("\n" + "-"*70)
    print("AGREGANDO DADOS POR TEMPORADA (LIGA)")
    print("-"*70)

    df_league = df.groupby('temporada')[numeric_cols].mean().reset_index()
    print(f"\nLigas identificadas: {len(df_league)}")
    print(df_league)

    # Normaliza as features
    scaler = StandardScaler()
    X = scaler.fit_transform(df_league[numeric_cols])

    print(f"\nDados normalizados (StandardScaler): shape={X.shape}")

    # 1. Método do Cotovelo para encontrar o k ótimo
    print("\n" + "-"*70)
    print("MÉTODO DO COTOVELO")
    print("-"*70)

    inertias = []
    silhouette_scores = []
    K_range = range(2, min(10, len(df_league)))

    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X)
        inertias.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(X, kmeans.labels_))
        print(f"k={k}: Inertia={kmeans.inertia_:.4f}, Silhouette={silhouette_scores[-1]:.4f}")

    # Plota Cotovelo e Silhueta
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Gráfico do Cotovelo
    axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
    axes[0].set_xlabel('Número de Clusters (k)', fontsize=12)
    axes[0].set_ylabel('Inertia (soma das distâncias ao centroide)', fontsize=12)
    axes[0].set_title('Método do Cotovelo', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xticks(K_range)

    # Gráfico da Silhueta
    axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
    axes[1].set_xlabel('Número de Clusters (k)', fontsize=12)
    axes[1].set_ylabel('Silhouette Score', fontsize=12)
    axes[1].set_title('Silhouette Score por k', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xticks(K_range)

    plt.tight_layout()
    plt.show()

    # Encontra o k ótimo (maior silhouette score)
    optimal_k = list(K_range)[np.argmax(silhouette_scores)]
    print(f"\n✓ K ótimo sugerido (por Silhouette Score): {optimal_k}")

    # 2. Aplica K-Means com k ótimo
    print("\n" + "-"*70)
    print(f"APLICANDO K-MEANS COM k={optimal_k}")
    print("-"*70)

    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X)

    df_league['cluster'] = clusters

    print(f"\nDistribuição de clusters:")
    print(df_league['cluster'].value_counts().sort_index())
    print(f"\nLigas por cluster:")
    for c in sorted(df_league['cluster'].unique()):
        leagues_in_cluster = df_league[df_league['cluster'] == c]['temporada'].tolist()
        print(f"  Cluster {c}: {leagues_in_cluster}")

    # 3. Visualiza clusters (PCA para visualização 2D)
    print("\n" + "-"*70)
    print("VISUALIZAÇÃO DOS CLUSTERS")
    print("-"*70)

    from sklearn.decomposition import PCA

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    print(f"\nVariância explicada pelos 2 primeiros componentes PCA: {pca.explained_variance_ratio_.sum():.2%}")

    plt.figure(figsize=(10, 8))
    colors = plt.cm.Set3(np.linspace(0, 1, optimal_k))

    for c in range(optimal_k):
        mask = clusters == c
        plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[colors[c]], label=f'Cluster {c}',
                   s=300, alpha=0.7, edgecolors='black', linewidth=2)

    # Plota centroides
    centroids_pca = pca.transform(kmeans.cluster_centers_)
    plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='red', marker='X', s=500,
               edgecolors='black', linewidth=2, label='Centroides', zorder=5)

    # Anota nomes das ligas
    for i, league in enumerate(df_league['temporada']):
        plt.annotate(league, (X_pca[i, 0], X_pca[i, 1]), fontsize=10, fontweight='bold',
                    xytext=(5, 5), textcoords='offset points')

    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=12)
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=12)
    plt.title(f'K-Means Clusterização de Ligas (k={optimal_k})', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 4. Análise da Silhueta para k ótimo
    print("\n" + "-"*70)
    print(f"ANÁLISE DE SILHUETA (k={optimal_k})")
    print("-"*70)

    silhouette_vals = silhouette_samples(X, clusters)

    fig, ax = plt.subplots(figsize=(10, 6))
    y_lower = 10
    colors = plt.cm.Set3(np.linspace(0, 1, optimal_k))

    for i in range(optimal_k):
        cluster_silhouette_vals = silhouette_vals[clusters == i]
        cluster_silhouette_vals.sort()

        size_cluster_i = cluster_silhouette_vals.shape[0]
        y_upper = y_lower + size_cluster_i

        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_vals,
                         facecolor=colors[i], edgecolor=colors[i], alpha=0.7, label=f'Cluster {i}')

        y_lower = y_upper + 10

    ax.set_xlabel('Coeficiente de Silhueta', fontsize=12)
    ax.set_ylabel('Cluster', fontsize=12)
    ax.set_title(f'Silhueta dos Clusters (k={optimal_k})', fontsize=14, fontweight='bold')
    ax.axvline(x=silhouette_score(X, clusters), color='red', linestyle='--', linewidth=2, label='Média')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

    print(f"\nSilhouette Score médio: {silhouette_score(X, clusters):.4f}")

    # 5. Características do Cluster
    print("\n" + "-"*70)
    print("CARACTERÍSTICAS DOS CLUSTERS")
    print("-"*70)

    for c in range(optimal_k):
        print(f"\n📍 CLUSTER {c}:")
        cluster_data = df_league[df_league['cluster'] == c]
        print(f"   Ligas: {cluster_data['temporada'].tolist()}")
        print(f"   Número de ligas: {len(cluster_data)}")
        print(f"\n   Características médias:")
        for col in numeric_cols:
            mean_val = cluster_data[col].mean()
            print(f"      {col}: {mean_val:.4f}")

    # 6. Salva resultados
    output_file = cleaned_file.parent / f"{cleaned_file.stem}_kmeans_clusters.csv"
    df_league.to_csv(output_file, index=False, encoding='utf-8')
    print(f"\n✓ Resultados salvos em: {output_file}")

else:
    print("Nenhum arquivo _cleaned.csv encontrado. Execute a célula de pré-processamento primeiro.")

## 🌍 Seção 8: Clusterização K-Means de Temporadas de TODAS as Ligas

Expandindo a análise anterior, esta seção aplica o K-Means para agrupar *todas as temporadas* de *todas as 15 ligas* em um único conjunto de dados. O objetivo é identificar padrões globais que transcendem ligas individuais, agrupando temporadas com características de jogo semelhantes independentemente de sua origem.

### **Processo:**
1.  **Carregamento e Consolidação:** Todos os arquivos `_cleaned.csv` das 15 ligas são carregados e concatenados.
2.  **Agregação por `liga` e `temporada`:** Calcula-se a média das features numéricas para cada combinação única de liga e temporada, criando um dataset onde cada linha representa uma temporada.
3.  **Normalização (`StandardScaler`):** As features são padronizadas para que a escala não influencie o clustering.
4.  **Determinação do `k` Ótimo:** O Método do Cotovelo e o Silhouette Score são usados para sugerir o número ideal de clusters.
    *   **Resultado do `k` Ótimo:** O Silhouette Score sugere `k=2` como o número ótimo de clusters.
5.  **Aplicação do K-Means:** O algoritmo é executado com o `k` ótimo.
6.  **Visualização (PCA):** A redução de dimensionalidade com PCA para 2 componentes permite visualizar a separação dos clusters, com rótulos de cada temporada (`liga (temporada)`).

### **Resultado Obtido (`k=2`):**
*   As 148 temporadas foram divididas em 2 clusters:
    *   **Cluster 0:** 59 temporadas.
    *   **Cluster 1:** 89 temporadas.
*   **Características Médias:** A análise das características médias de cada cluster (gols, odds) revela as diferenças que os separam. Por exemplo, o Cluster 0 tende a ter médias de `gols_casa` e `gols_fora` um pouco mais altas, com `odd_casa` e `odd_fora` também ligeiramente mais altas. Esta diferença, embora sutil, indica dois 'perfis' de temporadas.
*   **Silhouette Score Médio:** `0.2972`. Assim como na clusterização de ligas individuais, este score ainda é considerado moderado, sugerindo que a separação entre os dois clusters não é extremamente forte, mas existe uma estrutura.

### **É bom?**
*   Identificar dois perfis de temporadas para todas as ligas é um resultado interessante. Além disso, é possível notar uma predominância das ligas europeias no cluster 0, enquanto ligas americanas tendem a estar no cluster 1. Nesse contexto, é evidente uma melhor separação dos dados que será complementada pelas próximas tentativas de clusterizações.

In [ ]:
# ---- Clusterização K-Means: Temporadas de Todas as Ligas ----
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

# Lista de todas as ligas
leagues = ['argentina', 'belgica', 'brasileirao', 'bundesliga', 'colombia',
           'eredivisie', 'italia_seriaA', 'laliga', 'liga_portugal', 'ligue1',
           'mls', 'premier_league', 'russia', 'uruguai', 'venezuela']

data_dir = Path(".")

# Carrega todos os datasets limpos e agrega por temporada
print("="*70)
print("K-MEANS CLUSTERIZAÇÃO: TEMPORADAS DE TODAS AS LIGAS")
print("="*70)

dfs_season = []
for league in leagues:
    pattern = f"odds_{league}*_cleaned.csv"
    files = list(data_dir.glob(pattern))
    if files:
        file = files[0]
        try:
            df = pd.read_csv(file, encoding='utf-8')
            df['liga'] = league
            dfs_season.append(df)
            print(f"✓ {league:20s} - {len(df):5d} registros")
        except Exception as e:
            print(f"✗ {league:20s} - Erro: {e}")
    else:
        print(f"✗ {league:20s} - Arquivo não encontrado")

if dfs_season:
    df_all = pd.concat(dfs_season, ignore_index=True)
    print(f"\n{'='*70}")
    print(f"DATASET CONSOLIDADO: {len(df_all)} registros")
    print(f"{'='*70}")

    # Seleciona colunas numéricas para clusterização
    numeric_cols = df_all.select_dtypes(include=[np.number]).columns.tolist()
    for col in ['semana', 'odd_ratio']:
        if col in numeric_cols:
            numeric_cols.remove(col)

    print(f"\nFeatures para clusterização: {numeric_cols}")

    # Lida com valores ausentes
    df_all[numeric_cols] = df_all[numeric_cols].fillna(df_all[numeric_cols].median())

    # Agrega por temporada e liga
    print("\n" + "-"*70)
    print("AGREGANDO DADOS POR TEMPORADA E LIGA")
    print("-"*70)

    df_seasons = df_all.groupby(['liga', 'temporada'])[numeric_cols].mean().reset_index()
    df_seasons['season_label'] = df_seasons['liga'] + ' (' + df_seasons['temporada'].astype(str) + ')'

    print(f"\nTotal de temporadas encontradas: {len(df_seasons)}")
    print(f"\nAmostra das temporadas:")
    print(df_seasons[['liga', 'temporada', 'season_label']].head(20))

    # Normaliza as features
    scaler = StandardScaler()
    X = scaler.fit_transform(df_seasons[numeric_cols])

    print(f"\nDados normalizados (StandardScaler): shape={X.shape}")

    # 1. Método do Cotovelo
    print("\n" + "-"*70)
    print("MÉTODO DO COTOVELO")
    print("-"*70)

    inertias = []
    silhouette_scores = []
    K_range = range(2, min(15, len(df_seasons)//2))

    for k in K_range:
        kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans_temp.fit(X)
        inertias.append(kmeans_temp.inertia_)
        silhouette_scores.append(silhouette_score(X, kmeans_temp.labels_))
        print(f"k={k:2d}: Inertia={kmeans_temp.inertia_:.2f}, Silhouette={silhouette_scores[-1]:.4f}")

    # Plota Cotovelo e Silhueta
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
    axes[0].set_xlabel('Número de Clusters (k)', fontsize=12)
    axes[0].set_ylabel('Inertia', fontsize=12)
    axes[0].set_title('Método do Cotovelo - Temporadas de Todas as Ligas', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xticks(K_range)

    axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
    axes[1].set_xlabel('Número de Clusters (k)', fontsize=12)
    axes[1].set_ylabel('Silhouette Score', fontsize=12)
    axes[1].set_title('Silhouette Score - Temporadas de Todas as Ligas', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xticks(K_range)

    plt.tight_layout()
    plt.show()

    # Encontra o k ótimo
    optimal_k = list(K_range)[np.argmax(silhouette_scores)]
    print(f"\n✓ K ótimo sugerido (por Silhouette Score): {optimal_k}")

    # 2. Aplica K-Means com k ótimo
    print("\n" + "-"*70)
    print(f"APLICANDO K-MEANS COM k={optimal_k}")
    print("-"*70)

    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X)

    df_seasons['cluster'] = clusters

    print(f"\nDistribuição de clusters:")
    print(df_seasons['cluster'].value_counts().sort_index())

    # 3. Visualiza clusters com PCA
    print("\n" + "-"*70)
    print("VISUALIZAÇÃO DOS CLUSTERS (PCA 2D)")
    print("-"*70)

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    variance_explained = pca.explained_variance_ratio_.sum()
    print(f"\nVariância explicada pelos 2 primeiros componentes PCA: {variance_explained:.2%}")

    fig, ax = plt.subplots(figsize=(16, 12))
    colors = plt.cm.tab20(np.linspace(0, 1, optimal_k))

    for c in range(optimal_k):
        mask = clusters == c
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[colors[c]], label=f'Cluster {c}',
                  s=150, alpha=0.7, edgecolors='black', linewidth=1.5)

    # Plota centroides
    centroids_pca = pca.transform(kmeans.cluster_centers_)
    ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='red', marker='X', s=600,
              edgecolors='black', linewidth=2, label='Centroides', zorder=5)

    # Anota com rótulos de temporada
    for i, row in df_seasons.iterrows():
        ax.annotate(row['season_label'], (X_pca[i, 0], X_pca[i, 1]),
                   fontsize=7, ha='center', va='center',
                   bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                            alpha=0.6, edgecolor='gray', linewidth=0.5))

    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=12)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=12)
    ax.set_title(f'K-Means Clusterização de Temporadas (k={optimal_k})\nMostrando qual temporada de qual liga é similar a qual outra',
                fontsize=14, fontweight='bold')
    ax.legend(fontsize=9, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 4. Análise da Silhueta
    print("\n" + "-"*70)
    print(f"ANÁLISE DE SILHUETA (k={optimal_k})")
    print("-"*70)

    silhouette_vals = silhouette_samples(X, clusters)

    fig, ax = plt.subplots(figsize=(10, 10))
    y_lower = 10
    colors = plt.cm.tab20(np.linspace(0, 1, optimal_k))

    for i in range(optimal_k):
        cluster_silhouette_vals = silhouette_vals[clusters == i]
        cluster_silhouette_vals.sort()

        size_cluster_i = cluster_silhouette_vals.shape[0]
        y_upper = y_lower + size_cluster_i

        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_vals,
                         facecolor=colors[i], edgecolor=colors[i], alpha=0.7, label=f'Cluster {i}')

        y_lower = y_upper + 10

    ax.set_xlabel('Coeficiente de Silhueta', fontsize=12)
    ax.set_ylabel('Temporada', fontsize=12)
    ax.set_title(f'Silhueta dos Clusters (k={optimal_k})', fontsize=14, fontweight='bold')
    ax.axvline(x=silhouette_score(X, clusters), color='red', linestyle='--', linewidth=2, label='Média')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

    print(f"\nSilhouette Score médio: {silhouette_score(X, clusters):.4f}")

    # 5. Composição do Cluster
    print("\n" + "-"*70)
    print("COMPOSIÇÃO DOS CLUSTERS")
    print("-"*70)

    for c in range(optimal_k):
        cluster_data = df_seasons[df_seasons['cluster'] == c]
        print(f"\n📍 CLUSTER {c}: ({len(cluster_data)} temporadas)")
        print("   Temporadas neste cluster:")
        for _, row in cluster_data.iterrows():
            print(f"      • {row['season_label']}")

        print(f"\n   Características médias:")
        for col in numeric_cols:
            mean_val = cluster_data[col].mean()
            print(f"      {col}: {mean_val:.4f}")

    # 6. Distribuição por liga
    print("\n" + "-"*70)
    print("TEMPORADAS POR LIGA E CLUSTER")
    print("-"*70)
    cluster_league_table = pd.crosstab(df_seasons['liga'], df_seasons['cluster'], margins=True)
    print(cluster_league_table)

    # 7. Salva resultados
    output_file = data_dir / "all_leagues_seasons_kmeans_clusters.csv"
    df_seasons.to_csv(output_file, index=False, encoding='utf-8')
    print(f"\n✓ Resultados salvos em: {output_file}")

else:
    print("Nenhum arquivo _cleaned.csv encontrado para as ligas.")

## 🏟️ Seção 9: Clusterização K-Means de TODAS as Partidas em TODAS as Ligas

Nesta fase, a unidade de análise muda de 'temporada' para 'partida individual'. O K-Means é aplicado a *todas as partidas* (registros brutos, não agregados por temporada) de todas as 15 ligas. O objetivo é identificar diferentes 'tipos' de partidas que ocorrem no futebol, independentemente da liga ou temporada, focando em suas características de gols e odds.

### **Processo:**
1.  **Carregamento e Consolidação:** Todos os arquivos `_cleaned.csv` são concatenados, formando um dataset gigante de partidas.
2.  **Normalização (`StandardScaler`):** As features numéricas de cada partida são padronizadas.
3.  **Determinação do `k` Ótimo:** O Método do Cotovelo e o Silhouette Score são empregados. O Silhouette Score apontou `k=2` como o número ótimo de clusters.
4.  **Aplicação do K-Means:** O algoritmo é executado com `k=2` nos dados de todas as partidas.
5.  **Visualização (PCA):** A projeção PCA em 2D ajuda a visualizar a separação dos clusters de partidas.

### **Resultado Obtido (`k=2`):**
*   As 47.471 partidas foram divididas em 2 clusters:
    *   **Cluster 0:** 8.059 partidas.
    *   **Cluster 1:** 39.412 partidas.
*   **Características Médias dos Clusters:** A comparação das médias das features nos dois clusters revela distinções claras:
    *   **Cluster 0 (Gols mais altos para casa, odds de favorito mais baixas, odds de azarão mais altas):** Parece representar partidas onde o time da casa é um *grande favorito* e vence com uma *boa diferença de gols* (`gols_casa` médio de 2.89, `goal_diff` de 2.27, `odd_favorito` de 1.43, `odd_fora` de 10.39). Estes são resultados mais 'esperados' ou 'dominantes'.
    *   **Cluster 1 (Gols mais equilibrados, odds mais próximas):** Representa partidas mais *equilibradas*, com menor diferença de gols e odds mais altas para os favoritos, além de odds de empate/visitante mais baixas. (`gols_casa` médio de 1.22, `gols_fora` de 1.28, `goal_diff` de -0.05, `odd_favorito` de 2.08, `odd_fora` de 3.43). Inclui muitos empates e vitórias visitantes menos surpreendentes.
*   **Silhouette Score Médio:** `0.3861`. Este é um score razoável, indicando uma separação moderada dos clusters. É um valor melhor do que o obtido para as temporadas, sugerindo que as partidas individuais têm perfis mais distintos quando agrupadas desta forma.

### **É bom?**
*   Sim, é um resultado muito útil. A identificação de dois tipos principais de partidas (as 'dominantes' e as 'equilibradas') fornece insights valiosos sobre a natureza dos jogos de futebol. Pode ser usado, por exemplo, para desenvolver estratégias de apostas, segmentar análises táticas ou entender melhor a dinâmica das ligas. A capacidade de separar automaticamente esses padrões é um ponto forte da clusterização. Porém, a separação que havíamos visto na clusterização anterior já não é mais tão evidente nesse novo cenário, por isso, realizamos a visualização da próxima seção.

In [ ]:
# ---- Clusterização K-Means: TODAS as Partidas de Todas as Ligas ----
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

# Lista de todas as ligas
leagues = ['argentina', 'belgica', 'brasileirao', 'bundesliga', 'colombia',
           'eredivisie', 'italia_seriaA', 'laliga', 'liga_portugal', 'ligue1',
           'mls', 'premier_league', 'russia', 'uruguai', 'venezuela']

data_dir = Path(".")

# Carrega todos os datasets limpos
print("="*70)
print("CARREGANDO DATASETS DE TODAS AS LIGAS")
print("="*70)

dfs = []
for league in leagues:
    pattern = f"odds_{league}*_cleaned.csv"
    files = list(data_dir.glob(pattern))
    if files:
        file = files[0]
        try:
            df = pd.read_csv(file, encoding='utf-8')
            df['liga'] = league  # Adiciona o nome da liga como identificador
            dfs.append(df)
            print(f"✓ {league:20s} - {len(df):5d} registros")
        except Exception as e:
            print(f"✗ {league:20s} - Erro: {e}")
    else:
        print(f"✗ {league:20s} - Arquivo não encontrado")

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"\n{'='*70}")
    print(f"DATASET CONSOLIDADO: {len(df_all)} registros de {len(dfs)} ligas")
    print(f"{'='*70}")

    # Seleciona colunas numéricas para clusterização
    numeric_cols = df_all.select_dtypes(include=[np.number]).columns.tolist()
    for col in ['semana', 'temporada', 'odd_ratio']:
        if col in numeric_cols:
            numeric_cols.remove(col)

    print(f"\nFeatures para clusterização: {numeric_cols}")

    # Lida com valores ausentes
    df_all[numeric_cols] = df_all[numeric_cols].fillna(df_all[numeric_cols].median())

    print(f"Total de registros: {len(df_all)}")
    print(f"Distribuição por liga:")
    print(df_all['liga'].value_counts().sort_index())

    # Normaliza as features
    scaler = StandardScaler()
    X = scaler.fit_transform(df_all[numeric_cols])

    print(f"\nDados normalizados (StandardScaler): shape={X.shape}")

    # 1. Método do Cotovelo
    print("\n" + "-"*70)
    print("MÉTODO DO COTOVELO")
    print("-"*70)

    inertias = []
    silhouette_scores = []
    K_range = range(2, 11)

    for k in K_range:
        kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans_temp.fit(X)
        inertias.append(kmeans_temp.inertia_)
        silhouette_scores.append(silhouette_score(X, kmeans_temp.labels_))
        print(f"k={k}: Inertia={kmeans_temp.inertia_:.4f}, Silhouette={silhouette_scores[-1]:.4f}")

    # Plota Cotovelo e Silhueta
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
    axes[0].set_xlabel('Número de Clusters (k)', fontsize=12)
    axes[0].set_ylabel('Inertia', fontsize=12)
    axes[0].set_title('Método do Cotovelo', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xticks(K_range)

    axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
    axes[1].set_xlabel('Número de Clusters (k)', fontsize=12)
    axes[1].set_ylabel('Silhouette Score', fontsize=12)
    axes[1].set_title('Silhouette Score por k', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xticks(K_range)

    plt.tight_layout()
    plt.show()

    # Encontra o k ótimo
    optimal_k = list(K_range)[np.argmax(silhouette_scores)]
    print(f"\n✓ K ótimo sugerido (por Silhouette Score): {optimal_k}")

    # 2. Aplica K-Means com k ótimo
    print("\n" + "-"*70)
    print(f"APLICANDO K-MEANS COM k={optimal_k}")
    print("-"*70)

    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X)

    df_all['cluster'] = clusters

    print(f"\nDistribuição de clusters:")
    print(df_all['cluster'].value_counts().sort_index())

    print(f"\nLigas por cluster:")
    for c in sorted(df_all['cluster'].unique()):
        clusters_data = df_all[df_all['cluster'] == c]
        ligas_em_cluster = clusters_data['liga'].value_counts().to_dict()
        print(f"  Cluster {c}: {len(clusters_data)} partidas")
        for liga, count in sorted(ligas_em_cluster.items()):
            print(f"    - {liga}: {count} partidas")

    # 3. Visualiza clusters com PCA
    print("\n" + "-"*70)
    print("VISUALIZAÇÃO DOS CLUSTERS (PCA 2D)")
    print("-"*70)

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    variance_explained = pca.explained_variance_ratio_.sum()
    print(f"\nVariância explicada pelos 2 primeiros componentes PCA: {variance_explained:.2%}")

    fig, ax = plt.subplots(figsize=(14, 10))
    colors = plt.cm.tab10(np.linspace(0, 1, optimal_k))

    for c in range(optimal_k):
        mask = clusters == c
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[colors[c]], label=f'Cluster {c}',
                  s=30, alpha=0.6, edgecolors='none')

    # Plota centroides
    centroids_pca = pca.transform(kmeans.cluster_centers_)
    ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='red', marker='X', s=400,
              edgecolors='black', linewidth=2, label='Centroides', zorder=5)

    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=12)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=12)
    ax.set_title(f'K-Means Clusterização de Partidas de Todas as Ligas (k={optimal_k})',
                fontsize=14, fontweight='bold')
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 4. Análise da Silhueta
    print("\n" + "-"*70)
    print(f"ANÁLISE DE SILHUETA (k={optimal_k})")
    print("-"*70)

    silhouette_vals = silhouette_samples(X, clusters)

    fig, ax = plt.subplots(figsize=(10, 8))
    y_lower = 10
    colors = plt.cm.tab10(np.linspace(0, 1, optimal_k))

    for i in range(optimal_k):
        cluster_silhouette_vals = silhouette_vals[clusters == i]
        cluster_silhouette_vals.sort()

        size_cluster_i = cluster_silhouette_vals.shape[0]
        y_upper = y_lower + size_cluster_i

        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_vals,
                         facecolor=colors[i], edgecolor=colors[i], alpha=0.7, label=f'Cluster {i}')

        y_lower = y_upper + 10

    ax.set_xlabel('Coeficiente de Silhueta', fontsize=12)
    ax.set_ylabel('Cluster', fontsize=12)
    ax.set_title(f'Silhueta dos Clusters (k={optimal_k})', fontsize=14, fontweight='bold')
    ax.axvline(x=silhouette_score(X, clusters), color='red', linestyle='--', linewidth=2, label='Média')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

    print(f"\nSilhouette Score médio: {silhouette_score(X, clusters):.4f}")

    # 5. Características do Cluster
    print("\n" + "-"*70)
    print("CARACTERÍSTICAS DOS CLUSTERS")
    print("-"*70)

    for c in range(optimal_k):
        cluster_data = df_all[df_all['cluster'] == c]
        print(f"\n📍 CLUSTER {c}:")
        print(f"   Total de partidas: {len(cluster_data)}")
        print(f"   Ligas: {dict(cluster_data['liga'].value_counts())}")
        print(f"\n   Características médias:")
        for col in numeric_cols:
            mean_val = cluster_data[col].mean()
            print(f"      {col}: {mean_val:.4f}")

    # 6. Salva resultados
    output_file = data_dir / "all_leagues_kmeans_clusters.csv"
    df_all.to_csv(output_file, index=False, encoding='utf-8')
    print(f"\n✓ Resultados salvos em: {output_file}")

    # Exibe a distribuição do cluster por liga
    print("\n" + "-"*70)
    print("DISTRIBUIÇÃO DE CLUSTERS POR LIGA (TABELA)")
    print("-"*70)
    cluster_league_table = pd.crosstab(df_all['liga'], df_all['cluster'], margins=True)
    print(cluster_league_table)

else:
    print("Nenhum arquivo _cleaned.csv encontrado para as ligas.")

## 🗺️ Seção 10: Mapeamento de Similaridade entre Ligas (PCA)

Esta seção explora a similaridade entre as diferentes ligas de futebol através da análise de Componentes Principais (PCA) e cálculo de distâncias. O objetivo é visualizar como as ligas se posicionam umas em relação às outras em um espaço reduzido, baseado nas características médias de suas partidas.

### **Processo:**
1.  **Carregamento de Dados:** Todos os dados `_cleaned.csv` são consolidados.
2.  **Agregação por `liga`:** As features numéricas são agregadas por liga (calculando a média) para obter um 'perfil' médio de cada liga.
3.  **Normalização (`StandardScaler`):** As features agregadas são padronizadas.
4.  **PCA:** A Análise de Componentes Principais é aplicada para reduzir a dimensionalidade para 2 componentes (`PC1` e `PC2`), que capturam 62.51% da variância total.
5.  **Visualização:** Um gráfico de dispersão plota os centroides das ligas no espaço PCA, mostrando suas posições relativas.
6.  **Matriz de Distâncias:** As distâncias euclidianas entre os centroides das ligas são calculadas e exibidas em uma tabela e, posteriormente, em um mapa de calor.

### **Principais Insights:**
*   **Gráfico PCA:** Visualmente, podemos observar agrupamentos de ligas. Ligas geograficamente próximas ou com estilos de jogo historicamente semelhantes tendem a aparecer mais próximas no gráfico.
*   **Matriz de Distâncias:** Fornece um valor numérico da similaridade. Ligas com distâncias menores são mais parecidas em suas características de partida médias.
*   **Ligas Mais Similares:** O topo da lista mostra pares de ligas com perfis muito próximos (ex: 'belgica' ↔ 'ligue1', 'bundesliga' ↔ 'liga_portugal'), sugerindo similaridades no balanço de gols, odds, etc.
*   **Ligas Mais Diferentes:** O final da lista aponta para ligas com perfis muito distintos (ex: 'argentina' ↔ 'eredivisie'), indicando grandes contrastes nas características do jogo. Isso pode ocorrer devido a diferentes níveis de competitividade, filosofias táticas ou ambiente de apostas.

### **É bom?**
*   Este tipo de análise é extremamente valioso para entender o 'DNA' de cada liga, permitindo comparações objetivas. Ajuda a responder a pergunta **"Qual a proximidade (similaridade) entre as ligas, tanto em seus padrões gerais quanto nos seus eventos atípicos (outliers)?"** para o caso de padrões gerais. Pode informar analistas esportivos, casas de apostas (para transferir modelos ou estratégias entre ligas similares) e fãs interessados na natureza do esporte.

In [ ]:
# ---- Gráfico de Dispersão de Ligas no Espaço PCA ----
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Lista de todas as ligas
leagues = ['argentina', 'belgica', 'brasileirao', 'bundesliga', 'colombia',
           'eredivisie', 'italia_seriaA', 'laliga', 'liga_portugal', 'ligue1',
           'mls', 'premier_league', 'russia', 'uruguai', 'venezuela']

data_dir = Path(".")

# Carrega todos os datasets limpos
print("="*70)
print("CARREGANDO DATASETS PARA ANÁLISE DE DISTÂNCIAS ENTRE LIGAS")
print("="*70)

dfs = []
for league in leagues:
    pattern = f"odds_{league}*_cleaned.csv"
    files = list(data_dir.glob(pattern))
    if files:
        file = files[0]
        try:
            df = pd.read_csv(file, encoding='utf-8')
            df['liga'] = league
            dfs.append(df)
            print(f"✓ {league:20s} - {len(df):5d} registros")
        except Exception as e:
            print(f"✗ {league:20s} - Erro: {e}")
    else:
        print(f"✗ {league:20s} - Arquivo não encontrado")

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"\n{'='*70}")
    print(f"DATASET CONSOLIDADO: {len(df_all)} registros de {len(dfs)} ligas")
    print(f"{'='*70}")

    # Seleciona colunas numéricas
    numeric_cols = df_all.select_dtypes(include=[np.number]).columns.tolist()
    for col in ['semana', 'temporada', 'odd_ratio']:
        if col in numeric_cols:
            numeric_cols.remove(col)

    # Lida com valores ausentes
    df_all[numeric_cols] = df_all[numeric_cols].fillna(df_all[numeric_cols].median())

    # Normaliza as features
    scaler = StandardScaler()
    X = scaler.fit_transform(df_all[numeric_cols])

    # Aplica PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    # Calcula os centroides da liga (agrega por liga)
    league_centroids = []
    league_names = []

    for league in sorted(df_all['liga'].unique()):
        league_data = df_all[df_all['liga'] == league]
        league_indices = league_data.index

        # Obtém coordenadas PCA para esta liga
        league_pca = X_pca[league_indices]

        # Calcula o centroide
        centroid = league_pca.mean(axis=0)
        league_centroids.append(centroid)
        league_names.append(league)

    league_centroids = np.array(league_centroids)

    print("\n" + "="*70)
    print("DISPERSÃO DE LIGAS EM ESPAÇO PCA")
    print("="*70)
    print(f"\nVariância explicada pelos 2 primeiros componentes: {pca.explained_variance_ratio_.sum():.2%}")
    print(f"PC1: {pca.explained_variance_ratio_[0]:.2%}")
    print(f"PC2: {pca.explained_variance_ratio_[1]:.2%}")

    # Cria gráfico de dispersão
    fig, ax = plt.subplots(figsize=(16, 12))

    # Plota todos os pontos com transparência
    for league in sorted(df_all['liga'].unique()):
        league_data = df_all[df_all['liga'] == league]
        league_indices = league_data.index
        league_pca = X_pca[league_indices]
        ax.scatter(league_pca[:, 0], league_pca[:, 1], alpha=0.2, s=10, label=None)

    # Plota centroides da liga com cores distintas
    colors = plt.cm.tab20(np.linspace(0, 1, len(league_names)))
    for i, league_name in enumerate(league_names):
        ax.scatter(league_centroids[i, 0], league_centroids[i, 1],
                  c=[colors[i]], s=500, alpha=0.9, edgecolors='black',
                  linewidth=2, label=league_name, zorder=5)

        # Anota nomes das ligas
        ax.annotate(league_name, (league_centroids[i, 0], league_centroids[i, 1]),
                   fontsize=11, fontweight='bold', ha='center', va='center',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor=colors[i], alpha=0.7))

    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=13, fontweight='bold')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=13, fontweight='bold')
    ax.set_title('Dispersão de Ligas em Espaço PCA (Distância entre Ligas)',
                fontsize=15, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=9, loc='best', ncol=2, framealpha=0.9)

    plt.tight_layout()
    plt.show()

    # Calcula distâncias entre os centroides da liga
    print("\n" + "-"*70)
    print("MATRIZ DE DISTÂNCIAS EUCLIDIANAS ENTRE LIGAS")
    print("-"*70)

    from scipy.spatial.distance import pdist, squareform

    distances = squareform(pdist(league_centroids, metric='euclidean'))
    distances_df = pd.DataFrame(distances, index=league_names, columns=league_names)

    print("\nDistâncias (amostra de 5x5):")
    print(distances_df.iloc[:5, :5])

    # Encontra pares de ligas mais próximas e mais distantes
    print("\n" + "-"*70)
    print("LIGAS MAIS SIMILARES E MAIS DIFERENTES")
    print("-"*70)

    # Pega o triângulo superior para evitar duplicatas
    upper_triangle = np.triu_indices(len(league_names), k=1)
    distances_list = []

    for i, j in zip(upper_triangle[0], upper_triangle[1]):
        distances_list.append({
            'Liga 1': league_names[i],
            'Liga 2': league_names[j],
            'Distância': distances[i, j]
        })

    distances_sorted = sorted(distances_list, key=lambda x: x['Distância'])

    print("\n✓ TOP 10 LIGAS MAIS SIMILARES:")
    for idx, pair in enumerate(distances_sorted[:10], 1):
        print(f"  {idx:2d}. {pair['Liga 1']:20s} <-> {pair['Liga 2']:20s}: {pair['Distância']:.4f}")

    print("\n✗ TOP 10 LIGAS MAIS DIFERENTES:")
    for idx, pair in enumerate(distances_sorted[-10:], 1):
        print(f"  {idx:2d}. {pair['Liga 1']:20s} <-> {pair['Liga 2']:20s}: {pair['Distância']:.4f}")

    # Cria heatmap de distâncias
    fig, ax = plt.subplots(figsize=(14, 12))

    import seaborn as sns
    sns.heatmap(distances_df, annot=False, cmap='RdYlGn_r', square=True,
                cbar_kws={'label': 'Distância Euclidiana'}, ax=ax)

    ax.set_title('Matriz de Distâncias entre Ligas (PCA)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Liga', fontsize=12, fontweight='bold')
    ax.set_ylabel('Liga', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Estatísticas de distâncias
    print("\n" + "-"*70)
    print("ESTATÍSTICAS DE DISTÂNCIAS")
    print("-"*70)

    all_distances = distances_list
    distances_values = [d['Distância'] for d in all_distances]

    print(f"\nMínima distância: {min(distances_values):.4f}")
    print(f"Máxima distância: {max(distances_values):.4f}")
    print(f"Distância média: {np.mean(distances_values):.4f}")
    print(f"Desvio padrão: {np.std(distances_values):.4f}")
    print(f"Mediana: {np.median(distances_values):.4f}")

else:
    print("Nenhum arquivo _cleaned.csv encontrado para as ligas.")

## 🖼️ Seção 11: Visualização Detalhada da Distância entre Centroides de Ligas (PCA)

Esta seção refina a visualização da similaridade entre ligas. Em vez de plotar todos os pontos de dados individuais (que podem ser muitos e poluir o gráfico), ela foca apenas nos *centroides* de cada liga no espaço PCA. Além disso, ela adiciona linhas conectando ligas que estão abaixo de um certo limiar de distância, tornando a identificação de grupos de ligas similares mais intuitiva.

### **Principais Melhorias na Visualização:**
*   **Apenas Centroides:** Reduz a complexidade visual, apresentando apenas o 'perfil médio' de cada liga.
*   **Rótulos Aprimorados:** Os nomes das ligas são anotados diretamente nos pontos, com caixas de texto para maior clareza.
*   **Linhas de Conexão:** Linhas cinzas são desenhadas entre ligas cujos centroides estão a uma distância euclidiana inferior a 0.3. Esta distância de 0.3 atua como um 'limiar de similaridade', conectando ligas consideradas "próximas" ou "similares" com base nas características de suas partidas. O valor da distância é exibido na linha.
*   **Aspecto Igual:** Garante que a distância visual no gráfico corresponda fielmente à distância real no espaço PCA.

### **Análise Geográfica/Regional (Resumo):**
O resumo final tenta agrupar as ligas por regiões geográficas (Europeias, Sul-Americanas, Norte-Americana) e calcula a distância média de cada liga para outras ligas da mesma região (ou para todas as outras no caso da MLS). Isso ajuda a verificar se há uma tendência de ligas de uma mesma região serem mais similares entre si.

*   **Ligas Europeias:** As distâncias médias a outras ligas europeias variam, mas há clusters evidentes (ex: 'bundesliga' e 'liga_portugal' têm baixa distância média).
*   **Ligas Sul-Americanas:** Geralmente, as ligas sul-americanas mostram distâncias médias menores entre si, sugerindo um estilo de jogo ou padrão de odds mais coeso regionalmente.
*   **MLS (Norte-Americana):** Sua distância média para todas as outras ligas é maior, o que pode indicar um perfil de jogo mais único ou discrepante em relação às outras ligas analisadas. Embora sua proximidade esteja mais relacionada às ligas americanas, o que faz sentido para o perfil da liga.

In [ ]:
# ---- Gráfico de Dispersão Limpo: Apenas Centroides de Ligas com Distâncias ----
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform

# Lista de todas as ligas
leagues = ['argentina', 'belgica', 'brasileirao', 'bundesliga', 'colombia',
           'eredivisie', 'italia_seriaA', 'laliga', 'liga_portugal', 'ligue1',
           'mls', 'premier_league', 'russia', 'uruguai', 'venezuela']

data_dir = Path(".")

# Carrega todos os datasets limpos
dfs = []
for league in leagues:
    pattern = f"odds_{league}*_cleaned.csv"
    files = list(data_dir.glob(pattern))
    if files:
        file = files[0]
        try:
            df = pd.read_csv(file, encoding='utf-8')
            df['liga'] = league
            dfs.append(df)
        except Exception as e:
            pass
    else:
        pass

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)

    # Seleciona colunas numéricas
    numeric_cols = df_all.select_dtypes(include=[np.number]).columns.tolist()
    for col in ['semana', 'temporada', 'odd_ratio']:
        if col in numeric_cols:
            numeric_cols.remove(col)

    # Lida com valores ausentes
    df_all[numeric_cols] = df_all[numeric_cols].fillna(df_all[numeric_cols].median())

    # Normaliza as features
    scaler = StandardScaler()
    X = scaler.fit_transform(df_all[numeric_cols])

    # Aplica PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    # Calcula os centroides da liga
    league_centroids = []
    league_names = []

    for league in sorted(df_all['liga'].unique()):
        league_data = df_all[df_all['liga'] == league]
        league_indices = league_data.index
        league_pca = X_pca[league_indices]
        centroid = league_pca.mean(axis=0)
        league_centroids.append(centroid)
        league_names.append(league)

    league_centroids = np.array(league_centroids)

    # Cria um gráfico de dispersão limpo apenas com os centroides da liga
    fig, ax = plt.subplots(figsize=(16, 12))

    # Plota os centroides da liga com cores distintas
    colors = plt.cm.tab20(np.linspace(0, 1, len(league_names)))

    for i, league_name in enumerate(league_names):
        ax.scatter(league_centroids[i, 0], league_centroids[i, 1],
                  c=[colors[i]], s=400, alpha=0.85, edgecolors='black',
                  linewidth=2, label=league_name, zorder=5)

        # Anota os nomes das ligas abaixo do ponto
        ax.annotate(league_name, (league_centroids[i, 0], league_centroids[i, 1]),
                   fontsize=9, fontweight='bold', ha='center', va='top',
                   xytext=(0, -20), textcoords='offset points',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor=colors[i],
                            alpha=0.75, edgecolor='black', linewidth=1))

    # Desenha linhas conectando ligas próximas (distância < 0.3)
    distances = squareform(pdist(league_centroids, metric='euclidean'))

    for i in range(len(league_names)):
        for j in range(i+1, len(league_names)):
            dist = distances[i, j]
            if dist < 0.3:  # Apenas desenha linhas para ligas similares
                ax.plot([league_centroids[i, 0], league_centroids[j, 0]],
                       [league_centroids[i, 1], league_centroids[j, 1]],
                       color='gray', alpha=0.3, linewidth=1.5, zorder=1)

                # Adiciona rótulo de distância na linha
                mid_x = (league_centroids[i, 0] + league_centroids[j, 0]) / 2
                mid_y = (league_centroids[i, 1] + league_centroids[j, 1]) / 2
                ax.text(mid_x, mid_y, f'{dist:.3f}', fontsize=8,
                       bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                                alpha=0.7, edgecolor='gray', linewidth=0.5),
                       ha='center', va='center', zorder=3)

    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
                 fontsize=13, fontweight='bold')
    ax.set_title('Distâncias entre Ligas em Espaço PCA\n(Linhas conectam ligas similares, distância < 0.3)',
                fontsize=15, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.7)
    ax.legend(fontsize=10, loc='best', framealpha=0.95, ncol=3,
             title='Ligas', title_fontsize=11)

    # Define a proporção de aspecto igual para melhor visualização da distância
    ax.set_aspect('equal', adjustable='box')

    plt.tight_layout()
    plt.show()

    print("\n" + "="*70)
    print("MAPA DE DISTÂNCIAS ENTRE LIGAS (PCA)")
    print("="*70)
    print(f"\nVariância explicada:")
    print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
    print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")
    print(f"  Total: {pca.explained_variance_ratio_.sum():.2%}")


    # Cria uma tabela de resumo
    print("\n" + "="*70)
    print("RESUMO: LIGAS MAIS PRÓXIMAS POR REGIÃO")
    print("="*70)

    # Agrupa por clusters geográficos/regionais
    european = ['belgica', 'bundesliga', 'eredivisie', 'italia_seriaA', 'laliga', 'ligue1', 'liga_portugal', 'premier_league', 'russia']
    south_american = ['argentina', 'brasileirao', 'colombia', 'uruguai', 'venezuela']
    north_american = ['mls']

    print("\n🇪🇸 LIGAS EUROPEIAS:")
    for league in european:
        idx = league_names.index(league)
        avg_dist_to_euro = []
        for other in european:
            if other != league:
                other_idx = league_names.index(other)
                avg_dist_to_euro.append(distances[idx, other_idx])
        if avg_dist_to_euro:
            print(f"  {league:20s} - Distância média a outras europeias: {np.mean(avg_dist_to_euro):.4f}")

    print("\n🌎 LIGAS SUL-AMERICANAS:")
    for league in south_american:
        idx = league_names.index(league)
        avg_dist_to_sa = []
        for other in south_american:
            if other != league:
                other_idx = league_names.index(other)
                avg_dist_to_sa.append(distances[idx, other_idx])
        if avg_dist_to_sa:
            print(f"  {league:20s} - Distância média a outras sul-americanas: {np.mean(avg_dist_to_sa):.4f}")

    print("\n🌎 LIGA NORTE-AMERICANA:")
    mls_idx = league_names.index('mls')
    avg_dist_mls = np.mean([distances[mls_idx, league_names.index(l)] for l in league_names if l != 'mls'])
    print(f"  mls                  - Distância média a todas as outras: {avg_dist_mls:.4f}")

else:
    print("Nenhum arquivo _cleaned.csv encontrado para as ligas.")

## ⚠️ Seção 12: Clusterização K-Means de Temporadas (APENAS OUTLIERS)

Com a identificação de partidas *outliers* realizada na Seção 6, agora focamos em entender se as temporadas das ligas também apresentam perfis de "outlier". Ou seja, uma temporada inteira pode ser considerada atípica em relação às outras temporadas de outras ligas, se ela tiver uma quantidade ou tipo específico de eventos outliers. Esta seção repete o processo de clusterização K-Means, mas utilizando apenas as partidas que foram previamente identificadas como outliers (a partir dos arquivos `_outliers.csv`).

### **Processo:**
1.  **Carregamento de Dados de Outliers:** Todos os arquivos `*_outliers.csv` são carregados e concatenados.
2.  **Agregação por `liga` e `temporada`:** Calcula-se a média das features numéricas das partidas *outliers* para cada temporada, criando um 'perfil de outlier' por temporada.
3.  **Normalização (`StandardScaler`):** As features agregadas são padronizadas.
4.  **Determinação do `k` Ótimo:** O Método do Cotovelo e o Silhouette Score são usados para sugerir o número ideal de clusters.
    *   **Resultado do `k` Ótimo:** O Silhouette Score sugere `k=2` para agrupar as temporadas de outliers.
5.  **Aplicação do K-Means:** O algoritmo é executado com o `k` ótimo.
6.  **Visualização (PCA):** O PCA reduz a dimensionalidade para visualizar os clusters de temporadas-outliers. A variância explicada por 2 componentes foi de 65.21%.

### **Resultado Obtido (`k=2`):**
*   As 148 temporadas foram divididas em 2 clusters com base nas características médias de suas partidas outliers. O Cluster 0 tem 67 temporadas e o Cluster 1 tem 81 temporadas.
*   **Silhouette Score Médio:** `0.2835`. Este score, embora não seja alto, sugere que existem algumas distinções entre as temporadas quando analisamos apenas seus eventos outliers.
*   **Composição dos Clusters:** A tabela de *crosstab* (`liga` x `cluster`) mostra como as temporadas de cada liga se distribuem entre os clusters de outliers. Por exemplo, todas as temporadas da liga 'argentina' e 'colombia' ficaram no Cluster 0, enquanto todas as da 'bundesliga' e 'eredivisie' ficaram no Cluster 1. Isso indica que as *temporadas outliers* dessas ligas têm perfis diferentes.

### **É bom?**
*   Sim. Responder à pergunta **"Dentro das partidas consideradas 'outliers' (as zebras), existem subgrupos distintos de eventos surpreendentes?"** (aqui, a nível de temporada). É importante saber se certas ligas ou temporadas tendem a produzir um tipo específico de 'zebra' em comparação com outras. Por exemplo, um cluster pode ter outliers com odds de azarão extremamente altas (maiores zebras), enquanto outro tem outliers mais 'suaves'. Isso ajuda a entender a imprevisibilidade de cada liga.
* Contudo, ainda há a dificuldade para rotular o cluster, pois o conhecimento que temos é baseado em algumas partidas serem manipuladas e não a temporada por completo, então não podemos rotular diretamente os clusteres, necessitando uma nova análise.

In [ ]:
# ---- Clusterização K-Means: Temporadas (APENAS OUTLIERS) ----
# Esta célula repete a pipeline de clusterização (a partir da célula 11)
# mas usando apenas os arquivos que terminam com '_outliers.csv'.
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

data_dir = Path('.')
print('\n' + '='*70)
print('K-MEANS: TEMPORADAS COM APENAS _OUTLIERS.CSV')
print('='*70 + '\n')

# Encontra todos os arquivos *_outliers.csv
outlier_files = sorted(list(data_dir.glob('*_outliers.csv')))
if not outlier_files:
    print('Nenhum arquivo *_outliers.csv encontrado no diretório atual.')
else:
    print(f'Arquivos encontrados: {len(outlier_files)}')
    for f in outlier_files:
        print(' -', f.name)

    dfs = []
    for f in outlier_files:
        try:
            df = pd.read_csv(f, encoding='utf-8')
        except Exception:
            df = pd.read_csv(f, encoding='latin-1')
        # Se não há coluna 'liga', infere do nome do arquivo: odds_<liga>_..._outliers.csv
        if 'liga' not in df.columns:
            # tenta extrair entre 'odds_' e primeiro '_' após isso
            name = f.stem
            if name.startswith('odds_'):
                parts = name.split('_')
                # odds, league, maybe season, outliers etc.
                if len(parts) >= 2:
                    liga_inferida = parts[1]
                else:
                    liga_inferida = name
            else:
                liga_inferida = name
            df['liga'] = liga_inferida
        dfs.append(df)

    df_all_out = pd.concat(dfs, ignore_index=True)
    print('\nDataset consolidado (outliers):', len(df_all_out), 'registros')

    # Selecionar colunas numéricas relevantes
    numeric_cols = df_all_out.select_dtypes(include=[np.number]).columns.tolist()
    for c in ['semana','odd_ratio']:
        if c in numeric_cols:
            numeric_cols.remove(c)
    # Garante que temos algumas colunas numéricas
    if not numeric_cols:
        print('Não foram encontradas colunas numéricas para clusterização. Abortando.')
    else:
        print('\nColunas numéricas usadas para agregação/clusterização:')
        print(numeric_cols)

        # Preencher missing com mediana
        df_all_out[numeric_cols] = df_all_out[numeric_cols].fillna(df_all_out[numeric_cols].median())

        # Agrupar por (liga, temporada)
        if 'temporada' not in df_all_out.columns:
            print('\nA coluna "temporada" não foi encontrada; abortando agregação por temporada.')
        else:
            df_seasons_out = df_all_out.groupby(['liga','temporada'])[numeric_cols].mean().reset_index()
            df_seasons_out['season_label'] = df_seasons_out['liga'] + ' (' + df_seasons_out['temporada'].astype(str) + ')'
            print(f"\nTotal de temporadas (outliers) agregadas: {len(df_seasons_out)}")

            # Normalizar
            scaler = StandardScaler()
            X = scaler.fit_transform(df_seasons_out[numeric_cols])
            print('Dados normalizados: shape=', X.shape)

            # Elbow + Silhouette
            inertias = []
            silhouettes = []
            max_k = min(14, max(2, len(df_seasons_out)//2))
            K_range = range(2, max_k+1)
            print('\nExecutando Elbow + Silhouette para k em', list(K_range))
            for k in K_range:
                km = KMeans(n_clusters=k, random_state=42, n_init=10)
                km.fit(X)
                inertias.append(km.inertia_)
                silhouettes.append(silhouette_score(X, km.labels_))
                print(f' k={k}: inertia={km.inertia_:.2f}, silhouette={silhouettes[-1]:.4f}')

            # Plot
            fig, axs = plt.subplots(1,2,figsize=(14,5))
            axs[0].plot(list(K_range), inertias, 'o-', color='tab:blue')
            axs[0].set_title('Elbow (Outliers - Temporadas)')
            axs[0].set_xlabel('k')
            axs[0].set_ylabel('Inertia')
            axs[0].grid(alpha=0.3)

            axs[1].plot(list(K_range), silhouettes, 'o-', color='tab:green')
            axs[1].set_title('Silhouette (Outliers - Temporadas)')
            axs[1].set_xlabel('k')
            axs[1].set_ylabel('Silhouette Score')
            axs[1].grid(alpha=0.3)
            plt.tight_layout()
            plt.show()

            # Escolher k ótimo por Silhouette
            optimal_k = list(K_range)[int(np.argmax(silhouettes))]
            print('\nK ótimo (por Silhouette):', optimal_k)

            # Aplicar KMeans
            kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
            labels = kmeans.fit_predict(X)
            df_seasons_out['cluster'] = labels

            # PCA para visualizar
            pca = PCA(n_components=2)
            X_pca = pca.fit_transform(X)
            print('\nVariância explicada (2 componentes):', pca.explained_variance_ratio_.sum())

            fig, ax = plt.subplots(figsize=(14,10))
            colors = plt.cm.tab10(np.linspace(0,1,optimal_k))
            for c in range(optimal_k):
                mask = labels==c
                ax.scatter(X_pca[mask,0], X_pca[mask,1], c=[colors[c]], label=f'Cluster {c}', s=140, alpha=0.7, edgecolor='k')
            centroids_pca = pca.transform(kmeans.cluster_centers_)
            ax.scatter(centroids_pca[:,0], centroids_pca[:,1], c='red', marker='X', s=500, label='Centroides', edgecolor='k')
            for i,row in df_seasons_out.iterrows():
                ax.annotate(row['season_label'], (X_pca[i,0], X_pca[i,1]), fontsize=7, ha='center', va='center',
                           bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6, edgecolor='gray', linewidth=0.4))
            ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
            ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
            ax.set_title(f'KMeans - Temporadas (APENAS OUTLIERS) (k={optimal_k})')
            ax.legend()
            ax.grid(alpha=0.3)
            plt.tight_layout()
            plt.show()

            # Silhouette plot
            sil_vals = silhouette_samples(X, labels)
            fig, ax = plt.subplots(figsize=(8,8))
            y_lower = 10
            for i in range(optimal_k):
                ith_sil_vals = np.sort(sil_vals[labels==i])
                size_i = ith_sil_vals.shape[0]
                y_upper = y_lower + size_i
                ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_sil_vals, alpha=0.7)
                y_lower = y_upper + 10
            ax.axvline(np.mean(sil_vals), color='red', linestyle='--')
            ax.set_title('Silhueta - Temporadas (Outliers)')
            ax.set_xlabel('Coeficiente de Silhueta')
            ax.set_ylabel('Temporadas')
            plt.tight_layout()
            plt.show()

            print('\nDistribuição de clusters:')
            print(df_seasons_out['cluster'].value_counts().sort_index())

            # Mostrar composição por liga
            print('\nCrosstab liga x cluster:')
            print(pd.crosstab(df_seasons_out['liga'], df_seasons_out['cluster'], margins=True))

            # Salvar resultados
            out_file = data_dir / 'all_leagues_outliers_seasons_kmeans_clusters.csv'
            df_seasons_out.to_csv(out_file, index=False, encoding='utf-8')
            print('\n✓ Resultados salvos em', out_file)

## ⚽ Seção 13: Clusterização K-Means de Partidas Individuais (APENAS OUTLIERS)

Esta é uma análise crucial para entender a natureza das "zebras" no futebol. Em vez de agregar os dados por temporada, aqui tratamos cada **partida outlier individualmente** como um ponto de dados. O K-Means é aplicado a este conjunto de partidas outliers para ver se elas se agrupam em diferentes 'tipos' de eventos surpreendentes.

### **Processo:**
1.  **Carregamento de Dados:** Utiliza o dataset consolidado de *todas as partidas outliers* de todas as ligas (`df_all_out`).
2.  **Normalização (`StandardScaler`):** As features numéricas das partidas outliers são padronizadas.
3.  **Determinação do `k` Ótimo:** O Método do Cotovelo e o Silhouette Score são usados para encontrar o número ideal de clusters.
    *   **Resultado do `k` Ótimo:** O Silhouette Score sugere `k=2` como o número ótimo de clusters de partidas outliers.
4.  **Aplicação do K-Means:** O algoritmo é executado com `k=2`.

### **Resultado Obtido (`k=2`):**
*   As 4347 partidas outliers foram divididas em 2 clusters:
    *   **Cluster 0 (3771 partidas):** Representa o que o algoritmo considerou "outliers normais" ou menos extremos. Suas características médias indicam `odd_vencedora` de `5.4190`, `odd_favorito` de `1.5656` e `odd_preterido` de `6.5993`.
    *   **Cluster 1 (576 partidas):** Representa os "outliers extremos". Suas características médias mostram `odd_vencedora` muito mais alta (`9.9648`), `odd_favorito` mais baixa (`1.2440`) e `odd_preterido` significativamente maior (`14.6476`). Isso sugere que este cluster contém as "zebras das zebras", onde um time com uma odd baixíssima para vencer (ou empatar) perdeu para um time com odd altíssima.
*   **Silhouette Score Médio:** `0.4173`. Este é o melhor Silhouette Score obtido até agora, indicando uma boa separação entre os dois tipos de partidas outliers. É um forte indicador de que há duas categorias distintas de eventos surpreendentes.

### **É bom?**
*   **Sim, este é um resultado excelente!** Ele responde diretamente à pergunta **"Dentro das partidas consideradas 'outliers' (as zebras), existem subgrupos distintos de eventos surpreendentes?"** A separação em dois clusters (outliers 'normais' e outliers 'extremos') é um insight muito poderoso. Podemos agora entender e talvez até prever as condições que levam a diferentes níveis de "zebra". Isso tem aplicações diretas em análise de risco para apostas, entendimento da competitividade de ligas e storytelling esportivo.

In [ ]:
# ---- K-Means em todas as partidas (APENAS OUTLIERS) ----
# Cada linha é uma partida outlier; aplica KMeans e salva resultados.
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

data_dir = Path('.')

# Tenta carregar todos os *_outliers.csv concatenados (variável df_all_out pode existir)
if 'df_all_out' in globals():
    df_all_matches_out = df_all_out.copy()
    print('Usando df_all_out em memória (outliers)')
else:
    files = sorted(list(data_dir.glob('*_outliers.csv')))
    if not files:
        raise FileNotFoundError('Nenhum arquivo *_outliers.csv encontrado.')
    dfs = []
    for f in files:
        try:
            dfs.append(pd.read_csv(f, encoding='utf-8'))
        except Exception:
            dfs.append(pd.read_csv(f, encoding='latin-1'))
    df_all_matches_out = pd.concat(dfs, ignore_index=True)
    print('Carregados', len(files), 'arquivos outliers ->', len(df_all_matches_out), 'linhas')

# Seleciona colunas numéricas para clusterização (exclui semana e odd_ratio)
numeric_cols = df_all_matches_out.select_dtypes(include=[np.number]).columns.tolist()
for c in ['semana','odd_ratio']:
    if c in numeric_cols:
        numeric_cols.remove(c)
print('\nFeatures usadas (partidas outliers):', numeric_cols)

# Preencher NA
df_all_matches_out[numeric_cols] = df_all_matches_out[numeric_cols].fillna(df_all_matches_out[numeric_cols].median())

# Normalizar
scaler = StandardScaler()
X_matches = scaler.fit_transform(df_all_matches_out[numeric_cols])
print('Dados normalizados shape=', X_matches.shape)

# Elbow + Silhouette para faixa de k
inertias = []
sil_scores = []
max_k = min(12, max(2, len(df_all_matches_out)//50))
K_range = range(2, max_k+1)
print('\nExecutando Elbow+Silhouette para k em', list(K_range))
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_matches)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_matches, km.labels_))
    print(f' k={k}: inertia={km.inertia_:.2f}, silhouette={sil_scores[-1]:.4f}')

# Plots
fig, axs = plt.subplots(1,2,figsize=(14,5))
axs[0].plot(list(K_range), inertias, 'o-', color='tab:blue')
axs[0].set_title('Elbow (Outliers - Partidas)')
axs[0].set_xlabel('k')
axs[0].set_ylabel('Inertia')
axs[0].grid(alpha=0.3)
axs[1].plot(list(K_range), sil_scores, 'o-', color='tab:green')
axs[1].set_title('Silhouette (Outliers - Partidas)')
axs[1].set_xlabel('k')
axs[1].set_ylabel('Silhouette Score')
axs[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Escolhe o k ótimo por silhouette
optimal_k_matches = list(K_range)[int(np.argmax(sil_scores))]
print('\nK ótimo (partidas outliers):', optimal_k_matches)

# Ajusta KMeans com k ótimo e salva os rótulos
kmm = KMeans(n_clusters=optimal_k_matches, random_state=42, n_init=10)
labels_matches = kmm.fit_predict(X_matches)
df_all_matches_out['cluster'] = labels_matches

# Estatísticas por cluster
print('\nEstatísticas por cluster (partidas outliers):')
cluster_stats = df_all_matches_out.groupby('cluster')[numeric_cols].mean().round(4)
cluster_counts = df_all_matches_out['cluster'].value_counts().sort_index()
print(cluster_counts)
print(cluster_stats)

# Salva o resultado
out_matches_file = data_dir / 'all_leagues_outliers_matches_kmeans_clusters.csv'
df_all_matches_out.to_csv(out_matches_file, index=False, encoding='utf-8')
print('\n✓ Resultados salvos em', out_matches_file)

## 📈 Seção 14: Visualização PCA dos Clusters de Partidas Outliers

Para complementar a clusterização de partidas outliers, esta seção utiliza a Análise de Componentes Principais (PCA) para visualizar a separação dos clusters em um espaço 2D. Cada ponto no gráfico representa uma partida outlier individual, colorida de acordo com o cluster ao qual pertence.

### **Processo:**
1.  **Carregamento de Dados:** Utiliza o arquivo `all_leagues_outliers_matches_kmeans_clusters.csv` que já contém os rótulos de cluster para cada partida outlier.
2.  **Normalização (`StandardScaler`):** As features numéricas das partidas são novamente padronizadas para o PCA.
3.  **PCA (2 Componentes):** A dimensionalidade é reduzida para dois componentes principais (`PC1` e `PC2`).
    *   **Variância Explicada:** `60.41%` da variância total é capturada pelos dois primeiros componentes. Isso significa que o gráfico 2D representa uma parte significativa das diferenças originais nos dados.
4.  **Gráfico de Dispersão:** Os pontos são plotados, com cores distintas para cada cluster, e os rótulos indicam o número de partidas em cada cluster.

### **Principais Insights da Visualização:**
*   O gráfico mostra visualmente a separação dos dois clusters de partidas outliers. Embora possa haver alguma sobreposição, a tendência de agrupamento é perceptível, corroborando o Silhouette Score.
*   **Cluster 0 (Vermelho):** É o cluster maior, com `3771` partidas, representando os outliers 'normais' ou menos extremos.
*   **Cluster 1 (Cinza):** É o cluster menor, com `576` partidas, contendo os outliers 'extremos' ou as maiores zebras.

### **É bom?**
*   Sim, a visualização ajuda a confirmar graficamente a distinção entre os tipos de outliers identificados pelo K-Means. Ela oferece uma compreensão intuitiva de como esses dois grupos de partidas se diferenciam no espaço de features, tornando a análise mais robusta e comunicável.

In [ ]:
# ---- PCA Visualization: Clusters de Partidas Outliers ----
# Visualiza os clusters em PCA 2D (cada ponto é uma partida outlier)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from pathlib import Path

data_dir = Path('.')

# Carregar CSV com clusters ou usar variável em memória
matches_file = data_dir / 'all_leagues_outliers_matches_kmeans_clusters.csv'
if matches_file.exists():
    df_matches_clusters = pd.read_csv(matches_file, encoding='utf-8')
    print('Carregado arquivo:', matches_file)
elif 'df_all_matches_out' in globals():
    df_matches_clusters = df_all_matches_out.copy()
    print('Usando df_all_matches_out em memória')
else:
    raise FileNotFoundError('Arquivo all_leagues_outliers_matches_kmeans_clusters.csv não encontrado.')

# Selecionar colunas numéricas (mesmas usadas no clustering)
numeric_cols = df_matches_clusters.select_dtypes(include=[np.number]).columns.tolist()
for c in ['semana', 'odd_ratio', 'cluster']:
    if c in numeric_cols:
        numeric_cols.remove(c)

print('\nFeatures usadas para PCA:', numeric_cols)
print('Shape dos dados:', df_matches_clusters.shape)
print('Clusters únicos:', sorted(df_matches_clusters['cluster'].unique()))

# Normalizar (usar os dados normalizados anteriormente via StandardScaler)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_norm = scaler.fit_transform(df_matches_clusters[numeric_cols])

# Aplicar PCA
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_norm)

print(f'\nVariância explicada: {pca_2d.explained_variance_ratio_.sum():.2%}')
print(f'PC1: {pca_2d.explained_variance_ratio_[0]:.2%}')
print(f'PC2: {pca_2d.explained_variance_ratio_[1]:.2%}')

# Plotar
fig, ax = plt.subplots(figsize=(14, 10))

# Cores para cada cluster
clusters = sorted(df_matches_clusters['cluster'].unique())
colors = plt.cm.Set1(np.linspace(0, 1, len(clusters)))

# Scatter plot por cluster
for cluster_id, color in zip(clusters, colors):
    mask = df_matches_clusters['cluster'] == cluster_id
    count = mask.sum()
    ax.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
              c=[color], label=f'Cluster {cluster_id} (n={count})',
              s=50, alpha=0.6, edgecolors='k', linewidth=0.5)

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})', fontsize=12)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})', fontsize=12)
ax.set_title('K-Means Clusters de Partidas Outliers (PCA 2D)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Salvar figura
fig.savefig('outliers_matches_clusters_pca.png', dpi=150)
print('\n✓ Gráfico salvo em outliers_matches_clusters_pca.png')

## 📊 Seção 15: Análise da Distribuição de Ligas nos Clusters de Partidas Outliers

Após agrupar as partidas outliers em dois clusters distintos, é fundamental entender como as diferentes ligas contribuem para (ou se distribuem entre) esses clusters. Esta seção fornece uma análise detalhada da composição de cada cluster por liga, usando tabelas cruzadas e gráficos de barras.

### **Principais Insights:**
1.  **Contagens Absolutas:** Mostra o número exato de partidas outliers de cada liga em cada cluster.
2.  **Porcentagem por Liga (cada liga soma 100%):** Revela a proporção de partidas outliers de uma *determinada liga* que caíram em cada cluster. Por exemplo, a Eredivisie tem 76.76% de seus outliers no Cluster 0 e 23.24% no Cluster 1, indicando que uma parcela significativa de seus outliers são do tipo 'extremo'. A Uruguai, por outro lado, tem quase todos os seus outliers no Cluster 0 (98.89%).
3.  **Porcentagem por Cluster (qual % cada liga representa no cluster):** Indica a contribuição de cada liga para a formação de um cluster específico. Por exemplo, a Premier League contribui com 16.49% do total de partidas no Cluster 1 (outliers extremos), sendo a maior contribuinte.
4.  **Gráfico de Barras:** Visualiza claramente a distribuição percentual das ligas nos clusters, facilitando a comparação entre elas.

### **É bom?**
*   **Sim, é muito bom!** Esta análise responde diretamente à pergunta **"Qual a proximidade (similaridade) entre as ligas, tanto em seus padrões gerais quanto nos seus eventos atípicos (outliers)?"** especificamente para os outliers. Ligas que têm uma proporção maior de seus outliers no 'Cluster 1 (Extremo)' (como Premier League, Eredivisie, Laliga, Bundesliga) são, de certa forma, 'mais extremas' ou 'mais imprevisíveis' em seus resultados surpreendentes. Isso é um insight valioso para identificar ligas onde as "zebras" são não apenas mais frequentes, mas também mais impactantes ou inesperadas, com base nas odds. Contexto um tanto quanto inesperado devido às recorrentes notícias que recebemos sobre o envolvimento de bets no futebol brasileiro.

In [ ]:
# ---- Distribuição de Ligas por Cluster (Partidas Outliers) ----
# Análise detalhada: qual % de cada liga está em cada cluster
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

data_dir = Path('.')

# Carregar CSV com clusters ou usar variável em memória
matches_file = data_dir / 'all_leagues_outliers_matches_kmeans_clusters.csv'
if matches_file.exists():
    df_matches_clusters = pd.read_csv(matches_file, encoding='utf-8')
    print('Carregado arquivo:', matches_file)
elif 'df_all_matches_out' in globals():
    df_matches_clusters = df_all_matches_out.copy()
    print('Usando df_all_matches_out em memória')
else:
    raise FileNotFoundError('Arquivo all_leagues_outliers_matches_kmeans_clusters.csv não encontrado.')

print('\n' + '='*80)
print('DISTRIBUIÇÃO DE LIGAS POR CLUSTER (PARTIDAS OUTLIERS)')
print('='*80)

# Crosstab: ligas x clusters (contagens)
crosstab_counts = pd.crosstab(df_matches_clusters['liga'],
                               df_matches_clusters['cluster'],
                               margins=True)
print('\n📊 CONTAGENS ABSOLUTAS:')
print(crosstab_counts)

# Porcentagem dentro de cada liga (por linha)
crosstab_pct_by_liga = pd.crosstab(df_matches_clusters['liga'],
                                    df_matches_clusters['cluster'],
                                    normalize='index') * 100

print('\n📈 PORCENTAGEM POR LIGA (cada liga soma 100%):')
print(crosstab_pct_by_liga.round(2))

# Porcentagem dentro de cada cluster (por coluna)
crosstab_pct_by_cluster = pd.crosstab(df_matches_clusters['liga'],
                                       df_matches_clusters['cluster'],
                                       normalize='columns') * 100

print('\n🎯 PORCENTAGEM POR CLUSTER (qual % cada liga representa no cluster):')
print(crosstab_pct_by_cluster.round(2))

# Criar tabela mais legível com anotações
print('\n' + '='*80)
print('DETALHAMENTO POR LIGA:')
print('='*80)

for liga in sorted(df_matches_clusters['liga'].unique()):
    liga_data = df_matches_clusters[df_matches_clusters['liga'] == liga]
    total_liga = len(liga_data)

    print(f"\n{liga.upper()} (Total: {total_liga} partidas outliers)")
    print('-' * 60)

    for cluster_id in sorted(df_matches_clusters['cluster'].unique()):
        count_in_cluster = len(liga_data[liga_data['cluster'] == cluster_id])
        pct_in_liga = (count_in_cluster / total_liga) * 100
        pct_in_cluster = (count_in_cluster / len(df_matches_clusters[df_matches_clusters['cluster'] == cluster_id])) * 100

        print(f"  Cluster {cluster_id}: {count_in_cluster:4d} partidas ({pct_in_liga:6.2f}% da liga | {pct_in_cluster:5.2f}% do cluster)")

# Gráfico de barras: distribuição % de ligas em cada cluster
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Stacked bar (% dentro de cada liga)
crosstab_pct_by_liga.plot(kind='bar', stacked=False, ax=axes[0],
                          color=['#FF6B6B', '#4ECDC4'], width=0.7)
axes[0].set_title('Distribuição de Clusters por Liga\n(% dentro de cada liga)',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Liga', fontsize=11)
axes[0].set_ylabel('Porcentagem (%)', fontsize=11)
axes[0].legend(title='Cluster', labels=['Cluster 0 (Normal)', 'Cluster 1 (Extremo)'])
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# Subplot 2: Composição de cada cluster por liga (bar horizontal)
crosstab_pct_by_cluster_t = crosstab_pct_by_cluster.T
crosstab_pct_by_cluster_t.plot(kind='barh', stacked=True, ax=axes[1],
                               colormap='tab20', width=0.6)
axes[1].set_title('Composição de Cada Cluster por Liga\n(qual % cada liga representa)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Porcentagem (%)', fontsize=11)
axes[1].set_ylabel('Cluster', fontsize=11)
axes[1].legend(title='Liga', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Salvar tabelas em CSV para referência
crosstab_pct_by_liga.to_csv('outliers_liga_cluster_percentage_by_liga.csv', encoding='utf-8')
crosstab_pct_by_cluster.to_csv('outliers_liga_cluster_percentage_by_cluster.csv', encoding='utf-8')

print('\n✓ Tabelas salvas em:')
print('  - outliers_liga_cluster_percentage_by_liga.csv')
print('  - outliers_liga_cluster_percentage_by_cluster.csv')

## 📤 Seção 16: Exportação e Comparação dos Clusters de Partidas Outliers

Nesta seção final, as partidas outliers são separadas e salvas em arquivos CSV distintos para cada cluster, permitindo uma análise posterior mais fácil de cada tipo de evento. Além disso, são apresentadas estatísticas descritivas e uma comparação direta das médias das features entre os dois clusters.

### **Principais Saídas:**
1.  **Arquivos CSV por Cluster:**
    *   `outliers_cluster_0_normal.csv`: Contém 3771 partidas classificadas como outliers 'normais' ou menos extremos.
    *   `outliers_cluster_1_extremo.csv`: Contém 576 partidas classificadas como outliers 'extremos' ou as "maiores zebras".
2.  **Amostras dos Clusters:** Exibição das primeiras 5 linhas de cada arquivo para uma rápida visualização da composição.
3.  **Estatísticas Descritivas por Cluster:** Tabelas `describe()` para cada cluster, detalhando a distribuição estatística das features numéricas (média, desvio padrão, min/max, quartis).
4.  **Comparação de Médias:** Uma tabela comparativa mostra as médias de cada feature para ambos os clusters e a diferença entre elas. Esta é a maneira mais direta de entender o que define cada cluster.

### **Insights da Comparação de Médias:**
*   **`odd_vencedora` e `odd_preterido`**: São as features com as maiores diferenças, sendo significativamente maiores no Cluster 1 (extremo) em comparação com o Cluster 0 (normal). Isso confirma que o Cluster 1 realmente representa os resultados mais improváveis/surpreendentes.
*   **`odd_empate` e `odd_fora`**: Também são consideravelmente mais altas no Cluster 1, indicando que nesses jogos, as odds para empates e vitórias visitantes eram muito elevadas, confirmando o cenário de "zebra".
*   **`odd_favorito`**: É menor no Cluster 1, o que significa que nestas partidas houve um favorito muito claro (odd baixa) que acabou perdendo ou empatando contra todas as expectativas. Isso reforça a natureza 'extrema' desses outliers.
*   **`goal_diff`**: É menor no Cluster 1 (inclusive negativo em média), o que sugere que, embora fossem zebras, a diferença de gols não foi necessariamente enorme, mas sim que o resultado em si foi o mais surpreendente (ex: um 1x0 do azarão).

### **É bom?**
*   **Absolutamente!** Esta seção consolida as descobertas da clusterização de outliers. A capacidade de quantificar e comparar os dois tipos de eventos outliers é um grande passo para a compreensão da imprevisibilidade no futebol. Os arquivos CSV exportados podem ser usados para análises futuras mais direcionadas, como construir modelos preditivos específicos para cada tipo de outlier, ou investigar as condições (táticas, históricas, etc.) que levam a esses resultados.

In [ ]:
# ---- Exporta Partidas por Cluster (Outliers) ----
# Salva as partidas de cada cluster em arquivos separados
import pandas as pd
from pathlib import Path

data_dir = Path('.')

# Carrega CSV com clusters ou usa variável em memória
matches_file = data_dir / 'all_leagues_outliers_matches_kmeans_clusters.csv'
if matches_file.exists():
    df_matches_clusters = pd.read_csv(matches_file, encoding='utf-8')
    print('Carregado arquivo:', matches_file)
elif 'df_all_matches_out' in globals():
    df_matches_clusters = df_all_matches_out.copy()
    print('Usando df_all_matches_out em memória')
else:
    raise FileNotFoundError('Arquivo all_leagues_outliers_matches_kmeans_clusters.csv não encontrado.')

print('\n' + '='*80)
print('EXPORTANDO PARTIDAS POR CLUSTER')
print('='*80)

# Separa por cluster
cluster_0 = df_matches_clusters[df_matches_clusters['cluster'] == 0]
cluster_1 = df_matches_clusters[df_matches_clusters['cluster'] == 1]

print(f'\nCluster 0 (Normal): {len(cluster_0)} partidas')
print(f'Cluster 1 (Extremo): {len(cluster_1)} partidas')

# Salva em arquivos separados
file_cluster_0 = data_dir / 'outliers_cluster_0_normal.csv'
file_cluster_1 = data_dir / 'outliers_cluster_1_extremo.csv'

cluster_0.to_csv(file_cluster_0, index=False, encoding='utf-8')
cluster_1.to_csv(file_cluster_1, index=False, encoding='utf-8')

print(f'\n✓ Cluster 0 salvo em: {file_cluster_0}')
print(f'✓ Cluster 1 salvo em: {file_cluster_1}')

# Mostra amostra de cada cluster
print('\n' + '-'*80)
print('AMOSTRA: CLUSTER 0 (5 primeiras partidas)')
print('-'*80)
print(cluster_0[['liga', 'temporada', 'gols_casa', 'gols_fora', 'odd_casa', 'odd_empate',
                 'odd_fora', 'odd_vencedora', 'goal_diff', 'odd_favorito', 'odd_preterido']].head())

print('\n' + '-'*80)
print('AMOSTRA: CLUSTER 1 (5 primeiras partidas)')
print('-'*80)
print(cluster_1[['liga', 'temporada', 'gols_casa', 'gols_fora', 'odd_casa', 'odd_empate',
                 'odd_fora', 'odd_vencedora', 'goal_diff', 'odd_favorito', 'odd_preterido']].head())

# Estatísticas descritivas por cluster
print('\n' + '='*80)
print('ESTATÍSTICAS DESCRITIVAS POR CLUSTER')
print('='*80)

numeric_cols_stats = ['gols_casa', 'gols_fora', 'odd_casa', 'odd_empate', 'odd_fora',
                      'odd_vencedora', 'goal_diff', 'odd_favorito', 'odd_preterido']

print('\n📊 CLUSTER 0 (Normal):')
print(cluster_0[numeric_cols_stats].describe().round(4))

print('\n📊 CLUSTER 1 (Extremo):')
print(cluster_1[numeric_cols_stats].describe().round(4))

# Comparação: médias de cada cluster
print('\n' + '='*80)
print('COMPARAÇÃO: MÉDIAS POR CLUSTER')
print('='*80)

comparison_df = pd.DataFrame({
    'Cluster 0 (Normal)': cluster_0[numeric_cols_stats].mean(),
    'Cluster 1 (Extremo)': cluster_1[numeric_cols_stats].mean(),
    'Diferença (C1 - C0)': cluster_1[numeric_cols_stats].mean() - cluster_0[numeric_cols_stats].mean()
})

print('\n' + comparison_df.round(4).to_string())

# Salva comparação em CSV
comparison_file = data_dir / 'outliers_clusters_comparison.csv'
comparison_df.to_csv(comparison_file, encoding='utf-8')
print(f'\n✓ Comparação salva em: {comparison_file}')

## 📊 Seção 17: Características Médias dos Clusters de Temporadas Outliers

Esta seção tem como objetivo analisar e apresentar as características médias de cada cluster formado na análise de clusterização de *temporadas outliers*. Ao invés de partidas individuais, aqui estamos examinando os perfis médios das temporadas que foram consideradas atípicas.

### **Processo:**
1.  **Carregamento de Dados:** Utiliza o arquivo `all_leagues_outliers_seasons_kmeans_clusters.csv`, que contém as médias das features de outliers por temporada e o cluster ao qual cada temporada foi atribuída.
2.  **Agrupamento por Cluster:** Calcula-se a média de todas as features numéricas para cada cluster.

### **Principais Insights:**
*   A tabela `cluster_means` exibe o perfil médio de cada cluster de temporadas de outliers. Por exemplo, o Cluster 0 tem médias de `odd_vencedora` de `5.3921` e `odd_preterido` de `6.6109`, enquanto o Cluster 1 tem `odd_vencedora` de `6.3888` e `odd_preterido` de `8.2641`.
*   A `Contagem por cluster` mostra que o Cluster 0 contém 67 temporadas e o Cluster 1 contém 81 temporadas, indicando que a distribuição é relativamente equilibrada entre os dois tipos de 'perfil outlier' de temporada.
*   As diferenças nas médias das features entre os clusters nos dizem o que caracteriza uma temporada como pertencente a um grupo de outliers 'mais suaves' ou 'mais extremos' no agregado.

### **É bom?**
*   Sim, é importante para entender as **tendências de longo prazo** das ligas. Ligas que consistentemente caem em clusters de "temporadas com outliers mais extremos" podem ser consideradas mais imprevisíveis em seu histórico geral de zebras. Isso complementa a análise de partidas individuais e ajuda a responder **"Como as diferentes ligas e suas temporadas se agrupam com base nas características médias de suas partidas?"** quando o foco é apenas o comportamento outlier.

In [ ]:
# ---- Características médias dos clusters (Temporadas - Outliers) ----
# Imprime e salva as características médias por cluster (usa arquivo salvo ou variável em memória).
import pandas as pd
from pathlib import Path
import numpy as np

data_dir = Path('.')
seasons_file = data_dir / 'all_leagues_outliers_seasons_kmeans_clusters.csv'

if seasons_file.exists():
    df_seasons_out = pd.read_csv(seasons_file, encoding='utf-8')
    print('Carregado', seasons_file)
elif 'df_seasons_out' in globals():
    print('Usando df_seasons_out em memória')
else:
    raise FileNotFoundError('Arquivo all_leagues_outliers_seasons_kmeans_clusters.csv não encontrado e variável em memória ausente.')

# Detecta colunas numéricas para resumir (exclui coluna cluster)
numeric_cols = [c for c in df_seasons_out.columns if df_seasons_out[c].dtype.kind in 'ifu' and c not in ['cluster']]
print('\nColunas numéricas consideradas para as médias:')
print(numeric_cols)

cluster_means = df_seasons_out.groupby('cluster')[numeric_cols].mean().round(4)
cluster_counts = df_seasons_out.groupby('cluster').size()

print('\nCaracterísticas médias por cluster:')
print(cluster_means)
print('\nContagem por cluster:')
print(cluster_counts)

# Salva em CSV
out_char_file = data_dir / 'all_leagues_outliers_clusters_characteristics.csv'
cluster_means.reset_index().to_csv(out_char_file, index=False, encoding='utf-8')
print('\n✓ Características médias salvas em', out_char_file)

In [ ]:
# ---- K-Means em todas as partidas (APENAS OUTLIERS) ----
# Agrupa/usa todas as linhas outliers (cada partida outlier é um ponto); aplica KMeans e salva resultados.
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

data_dir = Path('.')
# Tenta carregar todos os *_outliers.csv concatenados (variável df_all_out pode existir)
if 'df_all_out' in globals():
    df_all_matches_out = df_all_out.copy()
    print('Usando df_all_out em memória (outliers)')
else:
    files = sorted(list(data_dir.glob('*_outliers.csv')))
    if not files:
        raise FileNotFoundError('Nenhum arquivo *_outliers.csv encontrado.')
    dfs = []
    for f in files:
        try:
            dfs.append(pd.read_csv(f, encoding='utf-8'))
        except Exception:
            dfs.append(pd.read_csv(f, encoding='latin-1'))
    df_all_matches_out = pd.concat(dfs, ignore_index=True)
    print('Carregados', len(files), 'arquivos outliers ->', len(df_all_matches_out), 'linhas')

# Seleciona colunas numéricas para clusterização (exclui semana e odd_ratio)
numeric_cols = df_all_matches_out.select_dtypes(include=[np.number]).columns.tolist()
for c in ['semana','odd_ratio']:
    if c in numeric_cols:
        numeric_cols.remove(c)
print('\nFeatures usadas (partidas outliers):', numeric_cols)

# Preencher NA
df_all_matches_out[numeric_cols] = df_all_matches_out[numeric_cols].fillna(df_all_matches_out[numeric_cols].median())

# Normalizar
scaler = StandardScaler()
X_matches = scaler.fit_transform(df_all_matches_out[numeric_cols])
print('\nDados normalizados shape=', X_matches.shape)

# Elbow + Silhouette para faixa de k
inertias = []
sil_scores = []
max_k = min(12, max(2, len(df_all_matches_out)//50))
K_range = range(2, max_k+1)
print('\nExecutando Elbow+Silhouette para k em', list(K_range))
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_matches)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_matches, km.labels_))
    print(f' k={k}: inertia={km.inertia_:.2f}, silhouette={sil_scores[-1]:.4f}')

# Plots
fig, axs = plt.subplots(1,2,figsize=(14,5))
axs[0].plot(list(K_range), inertias, 'o-', color='tab:blue')
axs[0].set_title('Elbow (Outliers - Partidas)')
axs[0].set_xlabel('k')
axs[0].set_ylabel('Inertia')
axs[0].grid(alpha=0.3)
axs[1].plot(list(K_range), sil_scores, 'o-', color='tab:green')
axs[1].set_title('Silhouette (Outliers - Partidas)')
axs[1].set_xlabel('k')
axs[1].set_ylabel('Silhouette Score')
axs[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Escolhe o k ótimo por silhouette
optimal_k_matches = list(K_range)[int(np.argmax(sil_scores))]
print('\nK ótimo (partidas outliers):', optimal_k_matches)

# Ajusta KMeans com k ótimo e salva os rótulos
kmm = KMeans(n_clusters=optimal_k_matches, random_state=42, n_init=10)
labels_matches = kmm.fit_predict(X_matches)
df_all_matches_out['cluster'] = labels_matches

# Estatísticas por cluster
print('\nEstatísticas por cluster (partidas outliers):')
cluster_stats = df_all_matches_out.groupby('cluster')[numeric_cols].mean().round(4)
cluster_counts = df_all_matches_out['cluster'].value_counts().sort_index()
print(cluster_counts)
print(cluster_stats)

# Salva o resultado
out_matches_file = data_dir / 'all_leagues_outliers_matches_kmeans_clusters.csv'
df_all_matches_out.to_csv(out_matches_file, index=False, encoding='utf-8')
print('\n✓ Resultados salvos em', out_matches_file)

## 📉 Seção 18: PCA e Matriz de Distâncias entre Centroides de Ligas (APENAS OUTLIERS)

Esta seção realiza uma análise de similaridade entre as ligas, mas focando exclusivamente nas características de suas **partidas outliers**. O objetivo é entender se a similaridade entre ligas muda quando consideramos apenas seus eventos mais surpreendentes, em contraste com a análise global feita anteriormente.

### **Processo:**
1.  **Carregamento de Dados:** Parte do dataset de *partidas outliers* (`df_all_matches_out`).
2.  **Agregação por `liga` (Centroides de Outliers):** Calcula a média das features numéricas das partidas outliers para cada liga, criando um 'centroide de outlier' para cada liga.
3.  **Normalização (`StandardScaler`):** Os centroides de outlier são padronizados.
4.  **PCA (2 Componentes):** A dimensionalidade é reduzida para 2 PCs, capturando **82.60%** da variância, um valor muito alto que indica que o PCA representa muito bem as diferenças originais.
5.  **Matriz de Distâncias:** As distâncias euclidianas são calculadas entre os centroides de outlier de cada liga.

### **Principais Insights:**
*   **Variância Explicada:** O PCA com dois componentes explica 82.60% da variância total, um valor excelente, indicando que o gráfico 2D e as distâncias capturam muito bem a complexidade original dos dados.
*   **Top 5 Pares Mais Semelhantes (Outliers):**
    *   'italia' ↔ 'laliga': 1.6918
    *   'argentina' ↔ 'venezuela': 1.7425
    *   'brasileirao' ↔ 'colombia': 1.7631
    *   'bundesliga' ↔ 'eredivisie': 1.9145
    *   'colombia' ↔ 'venezuela': 2.1156
    Estes são os pares de ligas que mais se assemelham no **tipo de partida outlier** que produzem.
*   **Top 5 Pares Mais Distintos (Outliers):**
    *   'eredivisie' ↔ 'venezuela': 7.2795
    *   'argentina' ↔ 'bundesliga': 7.8189
    *   'eredivisie' ↔ 'uruguai': 7.8350
    *   'bundesliga' ↔ 'uruguai': 8.1113
    *   'argentina' ↔ 'eredivisie': 8.2123
    Estes são os pares de ligas que são mais diferentes no **tipo de partida outlier** que produzem.

### **É bom?**
*   **Sim, muito bom!** Esta análise é crucial para responder **"Qual a proximidade (similaridade) entre as ligas, tanto em seus padrões gerais quanto nos seus eventos atípicos (outliers)?"** de uma perspectiva focada. Ela revela que as similaridades entre ligas podem ser diferentes quando olhamos apenas para seus eventos mais imprevisíveis. Uma liga pode ser globalmente similar a outra, mas ter um perfil de 'zebra' completamente distinto, o que é um insight valioso para estratégias especializadas (e.g., apostas de alto risco ou análise de imprevisibilidade).

In [ ]:
# ---- PCA e Matriz de Distâncias entre Centroides por Liga (Outliers) ----
# Calcula centroides por liga, aplica PCA em centroids e salva matriz de distâncias.
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

data_dir = Path('.')

# Carrega partidas outliers (cada linha é uma partida outlier)
if 'df_all_matches_out' in globals():
    df_matches = df_all_matches_out.copy()
    print('Usando df_all_matches_out em memória')
else:
    files = sorted(list(data_dir.glob('*_outliers.csv')))
    if not files:
        raise FileNotFoundError('Nenhum arquivo *_outliers.csv encontrado.')
    dfs = []
    for f in files:
        try:
            dfs.append(pd.read_csv(f, encoding='utf-8'))
        except Exception:
            dfs.append(pd.read_csv(f, encoding='latin-1'))
    df_matches = pd.concat(dfs, ignore_index=True)
    print('Carregados', len(files), 'arquivos ->', len(df_matches), 'linhas')

# Infere 'liga' se ausente
if 'liga' not in df_matches.columns:
    # Esta função é apenas um placeholder. A inferência real deve ser feita com base no nome do arquivo original.
    # Por exemplo: row['filename'].split('_')[1] se 'filename' for uma coluna existente.
    # Ou, se os arquivos '_outliers.csv' já contêm a 'liga', esta parte é desnecessária.
    def infer_liga_from_filename(row):
        # Exemplo simplificado, substitua pela lógica real se necessário
        return "liga_desconhecida"
    df_matches['liga'] = df_matches.apply(infer_liga_from_filename, axis=1)

# Seleciona colunas numéricas
numeric_cols = df_matches.select_dtypes(include=[np.number]).columns.tolist()
for c in ['semana','odd_ratio']:
    if c in numeric_cols:
        numeric_cols.remove(c)

if not numeric_cols:
    raise ValueError('Nenhuma coluna numérica disponível para agregação.')

print('\nColunas numéricas usadas para centroides:', numeric_cols)

# Agrupar por liga (centroide por liga)
league_centroids = df_matches.groupby('liga')[numeric_cols].mean()
league_centroids.reset_index(inplace=True)
print('\nCentroides por liga:')
print(league_centroids[['liga']])

# Escalar antes do PCA/distância
scaler = StandardScaler()
X_league = scaler.fit_transform(league_centroids[numeric_cols])

# PCA para 2 componentes
pca = PCA(n_components=2, random_state=42)
X_league_pca = pca.fit_transform(X_league)
print('\nVariância explicada por 2 PCs:', pca.explained_variance_ratio_.sum())

# Monta DataFrame de centroides com PCA
league_centroids['PC1'] = X_league_pca[:,0]
league_centroids['PC2'] = X_league_pca[:,1]

# Matriz de distâncias (euclidiana) entre centroides no espaço escalado
dist_matrix = cdist(X_league, X_league, metric='euclidean')
leagues = league_centroids['liga'].tolist()

dist_df = pd.DataFrame(dist_matrix, index=leagues, columns=leagues)

# Salva centroides e matriz
centroids_file = data_dir / 'league_centroids_outliers.csv'
dist_file = data_dir / 'league_centroids_distances_outliers.csv'
league_centroids.to_csv(centroids_file, index=False, encoding='utf-8')
dist_df.to_csv(dist_file, encoding='utf-8')

print('\n✓ Centroides salvos em', centroids_file)
print('✓ Matriz de distâncias salva em', dist_file)

# Mostra as 5 menores e maiores distâncias para uma visão rápida
pairs = []
for i in range(len(leagues)):
    for j in range(i+1, len(leagues)):
        pairs.append((leagues[i], leagues[j], dist_matrix[i,j]))
pairs_sorted = sorted(pairs, key=lambda x: x[2])
print('\nTop 5 pares mais semelhantes:')
for a,b,d in pairs_sorted[:5]:
    print(f' {a} ↔ {b}: {d:.4f}')
print('\nTop 5 pares mais distintos:')
for a,b,d in pairs_sorted[-5:]:
    print(f' {a} ↔ {b}: {d:.4f}')

## 🗺️ Seção 19: Visualização dos Centroides de Ligas (APENAS OUTLIERS) no Espaço PCA

Para tornar a análise de similaridade entre ligas (focada em outliers) mais visual e compreensível, esta seção gera um gráfico de dispersão dos centroides de outlier de cada liga no espaço PCA 2D.

### **Processo:**
1.  **Carregamento dos Centroides:** Utiliza o arquivo `league_centroids_outliers.csv`, que contém as coordenadas PC1 e PC2 dos centroides de outlier de cada liga.
2.  **Gráfico de Dispersão:** Plota cada centroide como um ponto, com o nome da liga anotado abaixo.

### **Principais Insights da Visualização:**
*   O gráfico oferece uma representação visual clara de quais ligas são mais próximas umas das outras em termos de seus padrões de outliers. Ligas que produzem tipos semelhantes de "zebras" aparecerão agrupadas.
*   É possível observar, por exemplo, que as ligas sul-americanas (argentina, brasileirao, colombia, venezuela, uruguai) tendem a se agrupar, sugerindo que seus eventos outliers têm características compartilhadas. Da mesma forma, as ligas europeias podem formar seus próprios agrupamentos.

### **É bom?**
*   Sim, esta visualização é uma excelente forma de resumir as complexas relações de similaridade entre ligas, focando no aspecto dos outliers. Ela ajuda a reforçar as descobertas da matriz de distâncias e permite uma identificação rápida de grupos de ligas com perfis de "zebra" semelhantes. Isso tem grande valor para análises comparativas e para a comunicação dos resultados a um público mais amplo.

In [ ]:
# ---- Gráfico de dispersão dos centroides por liga (PCA) ----
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Carrega centroides gerados
centroids_file = Path('league_centroids_outliers.csv')
if centroids_file.exists():
    centroids = pd.read_csv(centroids_file, encoding='utf-8')
else:
    raise FileNotFoundError('Arquivo league_centroids_outliers.csv não encontrado.')

fig, ax = plt.subplots(figsize=(12,10))

# Dispersão
ax.scatter(centroids['PC1'], centroids['PC2'], s=400, color='tab:blue', alpha=0.8, edgecolor='k')

# Anotações: nome da liga abaixo do ponto
for i,row in centroids.iterrows():
    ax.annotate(row['liga'], (row['PC1'], row['PC2']), xytext=(0,-18), textcoords='offset points', ha='center', fontsize=9)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Centroides por Liga (Outliers) - PCA 2D')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Opcional: salva figura
fig.savefig('league_centroids_outliers_pca_scatter.png', dpi=150)
print('\n✓ Scatter salvo em league_centroids_outliers_pca_scatter.png')

## 🕸️ Seção 20: Heatmap e Grafo de Rede de Distâncias entre Ligas (APENAS OUTLIERS)

Para finalizar a análise de similaridade entre ligas baseada em outliers, esta seção apresenta duas visualizações poderosas: um heatmap e um grafo de rede. Ambos fornecem diferentes perspectivas sobre as distâncias calculadas entre os centroides de outlier das ligas.

### **1. Heatmap da Matriz de Distâncias:**
*   **Visualização:** Um mapa de calor da matriz de distâncias euclidianas. As cores mais claras (verde/amarelo) indicam maior similaridade (menor distância), enquanto as cores mais escuras (vermelho) indicam maior dissimilaridade (maior distância).
*   **Insight:** Permite identificar rapidamente blocos de ligas similares e pares de ligas muito diferentes. Os valores numéricos nas células do heatmap quantificam essas distâncias.

### **2. Grafo de Rede de Proximidade:**
*   **Visualização:** Representa as ligas como 'nós' e as relações de similaridade (distância) como 'arestas'. Apenas arestas para ligas com distâncias menores que 1.5 vezes a mediana das distâncias são exibidas, para evitar um grafo muito denso e focar nas conexões mais fortes.
*   **Cores das Arestas:** Variam do verde ao vermelho, onde verde indica maior similaridade (menor peso/distância) e vermelho indica menor similaridade (maior peso/distância) entre os nós conectados. O layout Kamada-Kawai tenta posicionar os nós de forma que as distâncias no grafo correspondam às distâncias reais.
*   **Insight:** Oferece uma visão topológica das relações. Grupos de ligas muito conectadas e próximas no grafo formam "comunidades" de ligas com perfis de outliers semelhantes. Ligas isoladas ou com poucas conexões próximas são mais únicas em seus padrões de outliers.

### **Resumo das Ligas Mais Semelhantes e Distintas (Outliers):**
*   **🟢 Top 5 ligas MAIS SEMELHANTES (outliers):** Reitera os pares de ligas que têm as menores distâncias em seus perfis de outlier, como 'italia' ↔ 'laliga' ou 'argentina' ↔ 'venezuela'.
*   **🔴 Top 5 ligas MAIS DISTINTAS (outliers):** Destaca os pares com as maiores distâncias, mostrando quais ligas são as mais díspares em termos de seus eventos atípicos, como 'eredivisie' ↔ 'uruguai' ou 'argentina' ↔ 'eredivisie'.

### **É bom?**
*   **Extremamente útil!** Estas visualizações são a culminação da análise de similaridade de outliers, fornecendo uma compreensão aprofundada e acessível da pergunta **"Qual a proximidade (similaridade) entre as ligas, tanto em seus padrões gerais quanto nos seus eventos atípicos (outliers)?"**. O heatmap dá uma visão quantitativa e o grafo de rede uma visão qualitativa e estrutural. Juntos, eles permitem uma análise rica para comparar ligas, identificar "mercados" de similaridade e entender a diversidade de comportamentos de "zebra" no futebol mundial.

In [ ]:
pip install networkx

In [ ]:
# ---- Visualização de Distâncias entre Centroides por Liga (Heatmap + Rede) ----
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from pathlib import Path

data_dir = Path('.')

# Carrega matriz de distâncias
dist_file = data_dir / 'league_centroids_distances_outliers.csv'
if not dist_file.exists():
    raise FileNotFoundError(f'{dist_file} não encontrado.')

dist_df = pd.read_csv(dist_file, index_col=0, encoding='utf-8')
print('Matriz de distâncias carregada:', dist_df.shape)

# 1. HEATMAP
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(dist_df, annot=True, fmt='.2f', cmap='RdYlGn_r', cbar_kws={'label': 'Distância Euclidiana'},
            ax=ax, square=True, linewidths=0.5)
ax.set_title('Matriz de Distâncias entre Centroides por Liga (Outliers)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
fig.savefig('league_distances_heatmap_outliers.png', dpi=150)
print('✓ Heatmap salvo em league_distances_heatmap_outliers.png')

# 2. GRAFO DE REDE (arestas com pesos = distâncias)
fig, ax = plt.subplots(figsize=(14, 10))

# Cria grafo a partir da matriz de distâncias
G = nx.Graph()
leagues = dist_df.index.tolist()
G.add_nodes_from(leagues)

# Adiciona arestas (apenas distâncias < threshold para clareza)
threshold = np.percentile(dist_df.values[np.triu_indices_from(dist_df.values, k=1)], 50)
for i, l1 in enumerate(leagues):
    for j, l2 in enumerate(leagues):
        if i < j:
            d = dist_df.loc[l1, l2]
            if d < threshold * 1.5:  # Inclui arestas até 1.5x da mediana
                G.add_edge(l1, l2, weight=d)

# Layout Kamada-Kawai
pos = nx.kamada_kawai_layout(G, scale=2)

# Desenha nós
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=800, ax=ax, edgecolors='black', linewidths=2)

# Desenha arestas com cores baseadas em distância (menor = mais verde, maior = mais vermelho)
edges = G.edges()
weights = [G[u][v]['weight'] for u, v in edges]
edge_cmap = plt.cm.RdYlGn_r
edge_norm = plt.Normalize(vmin=min(weights), vmax=max(weights))

for (u, v), w in zip(edges, weights):
    ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]],
           color=edge_cmap(edge_norm(w)), linewidth=2, alpha=0.7, zorder=1)

# Desenha rótulos
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', ax=ax)

ax.set_title('Rede de Proximidade entre Ligas (Outliers)\n(Arestas = Distâncias < mediana)',
            fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()
fig.savefig('league_distances_network_outliers.png', dpi=150)
print('✓ Rede salvo em league_distances_network_outliers.png')

print('\n' + '='*70)
print('RESUMO DE DISTÂNCIAS (OUTLIERS)')
print('='*70)
# Encontra pares mais próximos e mais distantes
pairs = []
for i in range(len(leagues)):
    for j in range(i+1, len(leagues)):
        pairs.append((leagues[i], leagues[j], dist_df.iloc[i,j]))
pairs_sorted = sorted(pairs, key=lambda x: x[2])

print('\n🟢 Top 5 ligas MAIS SEMELHANTES (outliers):')
for a,b,d in pairs_sorted[:5]:
    print(f'   {a:15s} ↔ {b:15s}: {d:.4f}')

print('\n🔴 Top 5 ligas MAIS DISTINTAS (outliers):')
for a,b,d in pairs_sorted[-5:]:
    print(f'   {a:15s} ↔ {b:15s}: {d:.4f}')